# NB03 — Baselines: the reproduction gate

**Project:** CardioMamba-Net · **Stage:** 3 of 5
`01_verify` → `02_preprocess` → **`03_baselines`** → `04_cardiomamba_train` → `05_evaluate`

---

## Why this notebook exists

Before we are allowed to claim a new architecture wins, we have to show our pipeline can
**reproduce the numbers we are trying to beat**. This notebook re-implements all four networks
from Chowdhury et al. 2024 — FPN-1D, UNet-1D, LinkNet-1D and MultiResLinkNet — trains them on our
data with their objective, and puts our numbers next to theirs.

If MultiResLinkNet does not land near a temporal correlation of **61.9** on RVA combined, something
in our pipeline is wrong and **nothing after this point is trustworthy**. That is the gate.

One deliberate difference from their setup is declared in the paper:

- **Splits are strictly by subject** and test windows do not overlap (see NB02 §"leakage decision").
  Theirs almost certainly leaked. So our reproduction may land *below* their published figures —
  that is the expected direction, and it is the fair comparison for everything that follows.

Everything else is theirs: **1 input channel** (the arctangent-demodulated displacement `dy`),
ECG mapped to **[0, 1]**, 5 levels, 64 filters doubling, **plain MSE**, Adam at 5e-4,
1024-sample windows at 128 Hz.

---

## ⚠️ Accelerator: **GPU T4 × 2**

*Session options → Accelerator → **GPU T4 x2***, Internet **On**, `HF_TOKEN` secret attached.

Before the first NB03 session, run **`02b_cleanup_baselines_hf_once.ipynb`** on CPU and confirm
`RESET COMPLETE`. NB03 checks its remote receipt before starting any uploader or training work.

Before Run All, please click **+ Add Input → Notebook Output** and attach NB02's saved output.
That read-only Kaggle mount is the preferred corpus source; Hugging Face is only the fallback.

Both GPUs are used through `DataParallel`. AMP (mixed precision) is on, which roughly doubles
throughput on T4s and halves memory.

Every logical run is executed in a short-lived child process, five epochs at a time. When a
child exits, Linux reclaims its entire RAM allocation and CUDA context before the next chunk or
model starts. This is intentionally stronger than `gc.collect()` and prevents the checkpoint
serializer's native-memory growth from accumulating across a long Kaggle kernel.

## The run queue — this is how it survives Kaggle

There are **4 models × 4 experiment settings × 5 folds = 80 runs**. That does not fit in one
12-hour session, and it is not supposed to.

The notebook builds one **canonical queue**, checks which runs are already finished on Hugging
Face, and works through as many as fit in `TIME_BUDGET_H`. Worker processes are restarted
automatically; you do not need to restart the notebook between models. Then it pushes and stops cleanly.
**Start a new session and run it again** — it picks up exactly where it left off. Three or four
sessions completes the matrix. Every experiment/model/fold ID is created once and never
recomputed.

There is deliberately no saved quick-training run in this notebook. The forward/backward smoke
test checks all four models without creating a checkpoint; this prevents a 10-epoch trial from
being mixed into the 120-epoch scientific results.

## Cell-by-cell run guide

| Code cell | What runs | Typical time |
|---:|---|---:|
| 1 | Configuration | < 5 s |
| 2 | Imports, dependency, dual-GPU and disk checks | 1–3 min |
| 3 | Write/import the versioned shared libraries | 10–30 s |
| 4 | HF login; restore run summaries and resume markers | 1–5 min |
| 5 | Mount attached NB02 corpus or download fallback | < 1 min mounted; 3–12 min fallback |
| 6 | Forward/backward smoke test all four baselines | 2–8 min |
| 7 | Build the resumable run queue | < 10 s |
| 8 | Define split, dataset, and recording-safe evaluation helpers | < 10 s |
| 9 | Train/evaluate queued runs; checkpoint every epoch and at least every 5 min | up to 11.25 h/session |
| 10 | Assemble the reproduction table | < 1 min |
| 11 | Draw learning curves and comparison figures | 1–5 min |
| 12 | Write report/card and final upload | 2–15 min |

The training cell prints epoch time and ETA after its first epoch. **Full mode is deliberately a
multi-session queue**; each session uses up to 11.25 hours, pushes, and resumes on the next Run All.

---
# 1 · Configuration

In [1]:
CFG = {
    "SRC_REPO":  "Shanmuk4622/cr-rvs-radar-ecg-processed-v2", # NB02 output
    "DST_REPO":  "Shanmuk4622/cardiomamba-baselines-v2",       # isolated v2 runs
    "HF_PRIVATE": False,
    "RUN_ID":    "nb03_baselines_v3",
    "PROTOCOL_ID": "baseline-v3-subjectwise-5fold-dy-mse-target01",
    "RESET_RECEIPT": "cleanup_receipts/baselines-v2-canonical-reset-20260902.json",
    "REQUIRE_CLEAN_RESET": True,

    "WORK":    "/kaggle/working/nb03",
    "SCRATCH": "/kaggle/temp/nb03",
    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_UPLOADS_HOUR": 24,

    # ---- faithful reproduction of Chowdhury et al. 2024 ----------------------
    "CHANNELS":   ["dy"],        # THEIR input: one arctangent-demodulated displacement channel
    "BASE":       64,            # section 3.1: "initial layer containing 64 filters"
    "LEVELS":     4,             # 5 levels counting the bottleneck
    "LR":         5e-4,          # section 3.1
    "LOSS":       "mse",         # section 3.1: "As a loss function, the MSE function was used"
    "TARGET_01":  True,          # the paper's [0,1] target convention; now actually applied

    # ---- training ------------------------------------------------------------
    "EPOCHS":     120,
    "PATIENCE":   20,            # section 3.1
    "BATCH":      64,
    # With memory-mapped recordings, worker processes add almost no throughput here but
    # repeatedly retain pinned/shared host pages in long Jupyter sessions.  Keep loading in
    # the main process: measured data time is tiny compared with GPU compute, and RAM stays
    # bounded across the 80-run queue.
    "WORKERS":    0,
    "PIN_MEMORY": False,        # avoids a growing pinned-page pool in a long-lived kernel
    "HOST_RAM_WARN_GB": 18.0,   # warn and keep going; never break the experiment queue
    "WEIGHT_DECAY": 1e-4,
    "AMP":        True,
    "MULTI_GPU":  True,
    "REQUIRE_DUAL_T4": True,
    "SEED":       1337,
    "LOG_EVERY":  25,
    # Every observed epoch is well below five minutes and is checkpointed at its end.
    # Avoid serialising another full model two or three times inside the same short epoch.
    "CHECKPOINT_EVERY_STEPS": 1000,
    "CHECKPOINT_EVERY_S": 300,
    "EPOCHS_PER_PROCESS": 5,    # hard OS-level RAM reset at most every five epochs
    "PROCESS_ISOLATION": True,  # each chunk exits; Linux reclaims all worker RAM/CUDA state

    # ---- the queue -----------------------------------------------------------
    "MODELS":      ["fpn", "unet", "linknet", "multireslinknet"],
    "EXPERIMENTS": ["B_rva", "A_resting", "A_valsalva", "A_apnea"],   # B first: it is the headline
    "N_FOLDS":     5,
    "TIME_BUDGET_H": 11.25,      # use almost the whole 12 h session, then push cleanly
}
import json
print(json.dumps(CFG, indent=2))

{
  "SRC_REPO": "Shanmuk4622/cr-rvs-radar-ecg-processed-v2",
  "DST_REPO": "Shanmuk4622/cardiomamba-baselines-v2",
  "HF_PRIVATE": false,
  "RUN_ID": "nb03_baselines_v3",
  "PROTOCOL_ID": "baseline-v3-subjectwise-5fold-dy-mse-target01",
  "RESET_RECEIPT": "cleanup_receipts/baselines-v2-canonical-reset-20260902.json",
  "REQUIRE_CLEAN_RESET": true,
  "WORK": "/kaggle/working/nb03",
  "SCRATCH": "/kaggle/temp/nb03",
  "PUSH_INTERVAL_S": 1800,
  "HF_MAX_UPLOADS_HOUR": 24,
  "CHANNELS": [
    "dy"
  ],
  "BASE": 64,
  "LEVELS": 4,
  "LR": 0.0005,
  "LOSS": "mse",
  "TARGET_01": true,
  "EPOCHS": 120,
  "PATIENCE": 20,
  "BATCH": 64,
  "WORKERS": 0,
  "PIN_MEMORY": false,
  "HOST_RAM_WARN_GB": 18.0,
  "WEIGHT_DECAY": 0.0001,
  "AMP": true,
  "MULTI_GPU": true,
  "REQUIRE_DUAL_T4": true,
  "SEED": 1337,
  "LOG_EVERY": 25,
  "CHECKPOINT_EVERY_STEPS": 1000,
  "CHECKPOINT_EVERY_S": 300,
  "EPOCHS_PER_PROCESS": 5,
  "PROCESS_ISOLATION": true,
  "MODELS": [
    "fpn",
    "unet",
    "linknet",
 

In [2]:
import os, sys, gc, json, math, time, warnings, subprocess, platform, shutil, signal
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    import importlib.util
    miss = [x for x in p if importlib.util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=True)
        for x in miss: __import__(x.replace("-", "_"))
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd, torch
WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "runs", WORK / "results", WORK / "figures"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/2**30:.1f} GB  sm_{p.major}{p.minor}")
    torch.backends.cudnn.benchmark = True
else:
    print("  !! NO GPU -- set Accelerator to 'GPU T4 x2' in Session options.")
    print("     The notebook will still run on CPU but training will be impractically slow.")
if torch.cuda.device_count() < 2 or not all("T4" in torch.cuda.get_device_name(i)
                                            for i in range(torch.cuda.device_count())):
    raise RuntimeError("Select Kaggle Accelerator: GPU T4 x2, then restart and Run All.")

torch 2.10.0+cu128 | cuda True
  GPU0: Tesla T4  14.6 GB  sm_75
  GPU1: Tesla T4  14.6 GB  sm_75


---
# 2 · Library

Six modules, written to disk and imported. Five come straight from NB02's repo contract
(`crvs_sync`, `crvs_data`, `crvs_metrics`) or are defined here and reused by NB04 and NB05
(`crvs_models`, `crvs_losses`, `crvs_engine`).

`crvs_models.py` is the interesting one — it holds the four baseline architectures, including the
MultiRes block and ResPath that make MultiResLinkNet what it is.

In [3]:
MODULES = {
 "crvs_sync.py":   r"""
# crvs_sync.py -- conservative, resumable and interrupt-safe Hugging Face sync.
# A folder upload can involve several HTTP requests, so this deliberately schedules far
# fewer than the nominal API limit: one periodic upload per 30 minutes, plus major stages
# and a best-effort final upload on interrupt. All local writes are atomic.
import os, json, time, random, threading, atexit, signal
from collections import deque
from pathlib import Path
from datetime import datetime, timezone

SYNC_VERSION = 2

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

class RollingLimiter:
    # Limits upload_folder CALLS, not HTTP requests. The low default leaves a wide margin.
    def __init__(self, calls_per_hour=24, min_gap_s=20):
        self.limit = max(1, int(calls_per_hour)); self.min_gap = float(min_gap_s)
        self.times = deque(); self.lock = threading.Lock()
    def take(self, timeout=1800):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                while self.times and now - self.times[0] >= 3600:
                    self.times.popleft()
                gap = self.min_gap - (now - self.times[-1]) if self.times else 0.0
                window = 3600 - (now - self.times[0]) if len(self.times) >= self.limit else 0.0
                wait = max(0.0, gap, window)
                if wait <= 0:
                    self.times.append(now); return True
            if time.monotonic() + wait > deadline:
                return False
            time.sleep(min(wait, 10.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_upload_calls_hour=24,
                 retry_max=6, verbose=True):
        from huggingface_hub import HfApi
        if not token:
            raise RuntimeError("HF_TOKEN is missing. Add it under Kaggle > Add-ons > Secrets.")
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = max(300, int(push_interval_s))
        self.limiter = RollingLimiter(max_upload_calls_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = time.time(); self._dirty = threading.Event()
        self._force = threading.Event(); self._wake = threading.Event()
        self._stop = threading.Event(); self._upload_lock = threading.Lock()
        self._log_lock = threading.Lock(); self._dirty_lock = threading.Lock()
        self._dirty_generation = 0; self._before_final = None
        self._pushes = 0; self._failures = 0; self._closed = False
        self.history = self.local / "sync_history.jsonl"
        self.state_path = self.local / "sync_state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private,
                 interval_s=self.interval, sync_version=SYNC_VERSION)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def recently_pushed(self, seconds=10):
        return self._pushes > 0 and (time.time() - self._last_push) <= float(seconds)

    def log(self, event, _mark_dirty=True, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with self._log_lock, open(self.history, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, default=str) + "\n"); f.flush()
        except Exception:
            pass
        if _mark_dirty:
            self._touch()
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def _touch(self):
        with self._dirty_lock:
            self._dirty_generation += 1; self._dirty.set()

    def mark_dirty(self, reason=None):
        if reason:
            self.log("dirty", reason=reason)
        else:
            self._touch()

    def save_state(self, state):
        atomic_json(self.state_path, state); self._touch()

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text(encoding="utf-8"))
            except Exception as e:
                self.log("state_read_error", err=type(e).__name__)
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        # Download into the real working folder. Call before producing new local files.
        from huggingface_hub import snapshot_download
        target = Path(into or self.local); target.mkdir(parents=True, exist_ok=True)
        try:
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(target), allow_patterns=allow_patterns,
                                  max_workers=4)
            self.log("resume_pull_ok", _mark_dirty=False, path=str(p), patterns=allow_patterns)
            return True
        except Exception as e:
            self.log("resume_pull_empty", _mark_dirty=False,
                     err=f"{type(e).__name__}: {str(e)[:240]}")
            return False

    def set_before_final_flush(self, callback):
        # Trainer registers an atomic emergency-checkpoint callback while it is active.
        self._before_final = callback

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw)
        self._force.set(); self._wake.set()

    def _run_final_hook(self):
        cb = self._before_final
        if cb is not None:
            try:
                cb()
            except Exception as e:
                self.log("final_checkpoint_error", err=f"{type(e).__name__}: {e}")

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.limiter.take(timeout=1800):
                self.log("upload_call_limit_timeout"); return False
            try:
                info = upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                                     repo_type=self.repo_type, token=self.token,
                                     commit_message=msg,
                                     ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                                      "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                meta = {"last_push_utc": datetime.now(timezone.utc).isoformat(),
                        "pushes_this_session": self._pushes,
                        "last_commit": str(getattr(info, "oid", "")), "message": msg}
                atomic_json(self.local / "last_push.json", meta)
                self.log("push_ok", _mark_dirty=False, n=self._pushes,
                         commit=meta["last_commit"], msg=msg)
                return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {str(e)[:500]}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None, force=False, run_final_hook=False):
        if run_final_hook:
            self._run_final_hook()
        if not self._dirty.is_set() and not force:
            return True
        with self._upload_lock:
            if not self._dirty.is_set() and not force:
                return True
            with self._dirty_lock:
                generation = self._dirty_generation
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            label = "final" if final else "checkpoint"
            message = msg or f"{self.run_id} {label} @ {stamp}Z"
            ok = self._do_upload(message)
            if ok:
                self._force.clear()
                with self._dirty_lock:
                    if self._dirty_generation == generation:
                        self._dirty.clear()
            return ok

    def _loop(self):
        while not self._stop.is_set():
            remaining = max(1.0, self.interval - (time.time() - self._last_push))
            self._wake.wait(min(30.0, remaining)); self._wake.clear()
            if self._stop.is_set():
                break
            forced = self._force.is_set()
            due = (time.time() - self._last_push) >= self.interval
            if self._dirty.is_set() and (due or forced):
                try:
                    tag = "major-stage" if forced else "periodic-30min"
                    self.flush(msg=f"{self.run_id} {tag} @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=f"{type(e).__name__}: {e}")

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, force=True, run_final_hook=True,
                           msg=f"{self.run_id} interrupted (signal {signum})")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
                raise SystemExit(128 + int(signum))
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._closed:
            return
        self._closed = True; self.log("closing")
        self._stop.set(); self._wake.set()
        try:
            self._thread.join(timeout=5)
            self.flush(final=True, force=self._dirty.is_set(), run_final_hook=True)
        except Exception as e:
            print("Final Hugging Face sync failed:", type(e).__name__, e)
""",
 "crvs_data.py":   r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 4
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.seed = int(seed); self.epoch = 0
        self._cache = {}

    def set_epoch(self, epoch):
        # Augmentation is a pure function of (seed, epoch, index). With workers restarted
        # each epoch, an interrupted epoch can replay and skip batches byte-for-byte.
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            rng = np.random.RandomState(np.random.SeedSequence(
                [self.seed, self.epoch, int(i)]).generate_state(1)[0])
            if rng.rand() < 0.5:
                x = x + rng.randn(*x.shape).astype(np.float32) * 0.01
            if rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
""",
 "crvs_metrics.py": r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
""",
 "crvs_models.py": r"""
# crvs_models.py -- the four baseline 1-D segmentation networks.
# All four are standardised the way Chowdhury et al. 2024 describe (section 3.1):
# 5 levels, 64 filters in the first level, doubling thereafter. Input (B, C_in, 1024).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def cbr(i, o, k=3, s=1):
    return nn.Sequential(nn.Conv1d(i, o, k, s, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(cbr(i, o), cbr(o, o))
    def forward(self, x):
        return self.b(x)

# ------------------------------------------------------------------ UNet
class UNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList()
        for i in range(levels - 1):
            self.downs.append(DoubleConv(chs[i], chs[i + 1]))
        self.bott = DoubleConv(chs[-1], chs[-1] * 2)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        prev = chs[-1] * 2
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose1d(prev, c, 4, 2, 1))
            self.decs.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv1d(chs[0], out_ch, 1)
    def forward(self, x):
        skips = []
        h = self.inc(x); skips.append(h)
        for d in self.downs:
            h = d(F.max_pool1d(h, 2)); skips.append(h)
        h = self.bott(F.max_pool1d(h, 2))
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            h = up(h)
            if h.shape[-1] != sk.shape[-1]:
                h = F.interpolate(h, size=sk.shape[-1], mode="linear", align_corners=False)
            h = dec(torch.cat([h, sk], 1))
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ LinkNet
class LinkEnc(nn.Module):
    def __init__(self, i, o, stride=2):
        super().__init__()
        self.c1 = cbr(i, o, 3, stride)
        self.c2 = nn.Sequential(nn.Conv1d(o, o, 3, 1, 1, bias=False), nn.BatchNorm1d(o))
        self.sc = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))
    def forward(self, x):
        return F.relu(self.c2(self.c1(x)) + self.sc(x))

class LinkDec(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        m = max(i // 4, 8)
        self.a = cbr(i, m, 1)
        self.b = nn.Sequential(nn.ConvTranspose1d(m, m, 4, 2, 1, bias=False),
                               nn.BatchNorm1d(m), nn.ReLU(inplace=True))
        self.c = cbr(m, o, 1)
    def forward(self, x):
        return self.c(self.b(self.a(x)))

class LinkNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.bott = cbr(prev, prev)
        self.decs = nn.ModuleList()
        rev = list(reversed(chs))
        for k, c in enumerate(rev):
            nxt = rev[k + 1] if k + 1 < len(rev) else chs[0]
            self.decs.append(LinkDec(c, nxt))
        self.head = nn.Sequential(cbr(chs[0], chs[0]), nn.Conv1d(chs[0], out_ch, 1))
    def forward(self, x):
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 2 - k
            if j >= 0:
                s = skips[j]
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                h = h + s
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ FPN
class FPN1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, pyr=128):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.lat = nn.ModuleList([nn.Conv1d(c, pyr, 1) for c in chs])
        self.smooth = nn.ModuleList([cbr(pyr, pyr) for _ in chs])
        self.heads = nn.ModuleList([nn.Sequential(cbr(pyr, pyr // 2), cbr(pyr // 2, pyr // 2))
                                    for _ in chs])
        self.head = nn.Sequential(cbr(pyr // 2, pyr // 2), nn.Conv1d(pyr // 2, out_ch, 1))
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x); feats = []
        for e in self.encs:
            h = e(h); feats.append(h)
        ps = [None] * len(feats)
        ps[-1] = self.lat[-1](feats[-1])
        for i in range(len(feats) - 2, -1, -1):
            up = F.interpolate(ps[i + 1], size=feats[i].shape[-1], mode="linear",
                               align_corners=False)
            ps[i] = self.lat[i](feats[i]) + up
        ps = [s(p) for s, p in zip(self.smooth, ps)]
        acc = None
        for hd, p in zip(self.heads, ps):
            v = F.interpolate(hd(p), size=L, mode="linear", align_corners=False)
            acc = v if acc is None else acc + v
        return {"wave": torch.tanh(self.head(acc))}

# ------------------------------------------------------------------ MultiResLinkNet
class MultiResBlock(nn.Module):
    # MultiResUNet block (Ibtehaz & Rahman) in 1-D: three successive 3-conv stages of
    # increasing width, concatenated, plus a 1x1 residual shortcut.
    def __init__(self, cin, U, alpha=1.67):
        super().__init__()
        W = alpha * U
        # max(1, ...): below U=4 the 0.167 stage floors to zero channels, and the failure
        # then surfaces as an opaque conv error rather than pointing here.
        c1, c2, c3 = (max(1, int(W * 0.167)), max(1, int(W * 0.333)), max(1, int(W * 0.5)))
        self.out_channels = c1 + c2 + c3
        self.sc = nn.Sequential(nn.Conv1d(cin, self.out_channels, 1, bias=False),
                                nn.BatchNorm1d(self.out_channels))
        self.a = cbr(cin, c1); self.b = cbr(c1, c2); self.c = cbr(c2, c3)
        self.bn1 = nn.BatchNorm1d(self.out_channels)
        self.bn2 = nn.BatchNorm1d(self.out_channels)
    def forward(self, x):
        s = self.sc(x)
        a = self.a(x); b = self.b(a); c = self.c(b)
        o = self.bn1(torch.cat([a, b, c], 1))
        return F.relu(self.bn2(o + s))

class ResPath(nn.Module):
    # Processes an encoder feature before it is added to the decoder, instead of a raw skip.
    def __init__(self, ch, length):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(max(1, length)):
            self.blocks.append(nn.ModuleDict({
                "sc": nn.Sequential(nn.Conv1d(ch, ch, 1, bias=False), nn.BatchNorm1d(ch)),
                "cv": nn.Sequential(nn.Conv1d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm1d(ch)),
            }))
    def forward(self, x):
        for b in self.blocks:
            x = F.relu(b["sc"](x) + b["cv"](x))
        return x

class MultiResLinkNet1D(nn.Module):
    # LinkNet skeleton, MultiRes blocks instead of plain convolutions, ResPath skips added
    # (not concatenated), and deep supervision from every encoder level.
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        units = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        self.bott = MultiResBlock(prev, units[-1])
        self.decs = nn.ModuleList()
        rev_ch = list(reversed(enc_ch))
        cur = self.bott.out_channels
        # Decoder step k must emerge with the channel count AND length of skips[-1-k], or
        # the additive skip is silently dropped and every ResPath receives zero gradient.
        # Encoder here pools AFTER appending the skip, so the target is rev_ch[k] -- not
        # rev_ch[k+1], which is correct only for the stride-2 encoder in LinkNet1D.
        for k in range(levels):
            tgt = rev_ch[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.head = nn.Sequential(cbr(cur, base), nn.Conv1d(base, out_ch, 1))
        self.aux = nn.ModuleList([nn.Conv1d(c, out_ch, 1) for c in enc_ch]) \
                   if deep_supervision else None
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(                     # raise, not assert: an invariant
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs "        # this
                        f"{s.shape[1]}. Dropping it silently is what cost 59% of this "  # load
                        "model's gradient once already.")   # bearing must survive python -O
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {"wave": torch.tanh(self.head(h))}
        if self.aux is not None and self.training:
            out["aux"] = [F.interpolate(a(s), size=L, mode="linear", align_corners=False)
                          for a, s in zip(self.aux, skips)]
        return out

BASELINES = {"fpn": FPN1D, "unet": UNet1D, "linknet": LinkNet1D,
             "multireslinknet": MultiResLinkNet1D}

def build_baseline(name, in_ch=1, out_ch=1, base=64, levels=4):
    return BASELINES[name](in_ch=in_ch, out_ch=out_ch, base=base, levels=levels)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
""",
 "crvs_losses.py": r"""
# crvs_losses.py -- C5, the morphology-aware composite loss.
# Plain MSE is the conditional mean, so it flattens the R peak; that is exactly why the
# baseline over-estimates RMSSD by ~2x. Every term here exists to stop that.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiResSTFTLoss(nn.Module):
    # Spectral convergence + log-magnitude at three resolutions. Forces the model to get
    # the spectrum right, not just the sample-wise average.
    def __init__(self, ffts=(256, 128, 64)):
        super().__init__()
        self.ffts = ffts
    def _one(self, y, yh, n):
        hop, win = n // 4, n
        w = torch.hann_window(win, device=y.device, dtype=torch.float32)
        kw = dict(n_fft=n, hop_length=hop, win_length=win, window=w,
                  return_complex=True, center=True, pad_mode="reflect")
        Y = torch.stft(y, **kw).abs().clamp_min(1e-7)
        H = torch.stft(yh, **kw).abs().clamp_min(1e-7)
        sc = torch.norm(Y - H, p="fro", dim=(-2, -1)) / (torch.norm(Y, p="fro", dim=(-2, -1)) + 1e-7)
        mag = F.l1_loss(torch.log(H), torch.log(Y))
        return sc.mean() + mag
    def forward(self, y, yh):
        y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
        return sum(self._one(y, yh, n) for n in self.ffts) / len(self.ffts)

def pearson_loss(y, yh, eps=1e-8):
    y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
    y = y - y.mean(-1, keepdim=True); yh = yh - yh.mean(-1, keepdim=True)
    num = (y * yh).sum(-1)
    den = y.norm(dim=-1) * yh.norm(dim=-1) + eps
    return (1.0 - num / den).mean()

def focal_bce(logit, target, alpha=0.75, gamma=2.0):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pt = p * target + (1 - p) * (1 - target)
    w = alpha * target + (1 - alpha) * (1 - target)
    return (w * (1 - pt).pow(gamma) * ce).mean()

class CompositeLoss(nn.Module):
    def __init__(self, w_huber=1.0, w_stft=0.5, w_peak=0.3, w_rr=0.1,
                 w_peakw=0.5, w_corr=0.3, huber_delta=0.1, peak_weight=4.0):
        super().__init__()
        self.w = dict(huber=w_huber, stft=w_stft, peak=w_peak, rr=w_rr,
                      peakw=w_peakw, corr=w_corr)
        self.delta = huber_delta
        self.peak_weight = peak_weight
        self.stft = MultiResSTFTLoss()
    def forward(self, pred, y, pk=None, rr=None):
        parts = {}
        wave = pred["wave"]
        if self.w["huber"]:
            parts["huber"] = F.huber_loss(wave, y, delta=self.delta)
        if self.w["stft"]:
            parts["stft"] = self.stft(y, wave)
        if self.w["corr"]:
            parts["corr"] = pearson_loss(y, wave)
        if self.w["peakw"] and pk is not None:
            wgt = 1.0 + self.peak_weight * pk
            parts["peakw"] = ((wgt * (wave - y).abs()).sum() / (wgt.sum() + 1e-8))
        if self.w["peak"] and pk is not None and "peak" in pred:
            parts["peak"] = focal_bce(pred["peak"], pk)
        if self.w["rr"] and rr is not None and "rr" in pred:
            parts["rr"] = F.l1_loss(pred["rr"], rr)
        if "aux" in pred:
            parts["aux"] = sum(F.huber_loss(a, y, delta=self.delta)
                               for a in pred["aux"]) / max(len(pred["aux"]), 1) * 0.2
        total = sum(self.w.get(k, 1.0) * v for k, v in parts.items())
        return total, {k: float(v.detach()) for k, v in parts.items()}

class MSEOnly(nn.Module):
    # The baseline's objective, kept verbatim so ablation row 1 is a true reproduction.
    def forward(self, pred, y, pk=None, rr=None):
        l = F.mse_loss(pred["wave"], y)
        if "aux" in pred:
            l = l + 0.2 * sum(F.mse_loss(a, y) for a in pred["aux"]) / max(len(pred["aux"]), 1)
        return l, {"mse": float(l.detach())}
""",
 "crvs_engine.py": r"""
# crvs_engine.py -- deterministic dual-GPU engine with mid-epoch recovery and telemetry.
import csv, json, math, time, os, random, shutil, subprocess, threading, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ENGINE_VERSION = 4

def _autocast(device_type, enabled):
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def _grad_scaler(device_type, enabled):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def pick_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        return torch.device("cuda"), n, names
    return torch.device("cpu"), 0, []

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def _atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str, allow_nan=True)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def _jsonable(v):
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, float) and not math.isfinite(v): return None
    return v

def _append_jsonl(path, record):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    clean = {str(k): _jsonable(v) for k, v in record.items()}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(clean, default=str, allow_nan=False) + "\n"); f.flush()

def _mean_parts(sums, n, prefix):
    return {f"{prefix}_{k}": float(v) / max(int(n), 1) for k, v in sums.items()}

def _system_stats(out_dir):
    d = shutil.disk_usage(Path(out_dir))
    rec = {"disk_free_gb": d.free / 2**30, "disk_used_gb": d.used / 2**30}
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        rec["process_peak_rss_gb"] = rss / 2**20  # Linux ru_maxrss is KiB
    except Exception:
        pass
    try:
        load = os.getloadavg(); rec.update(cpu_load_1m=load[0], cpu_load_5m=load[1], cpu_load_15m=load[2])
    except Exception:
        pass
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            rec[f"gpu{i}_peak_alloc_gb"] = torch.cuda.max_memory_allocated(i) / 2**30
            rec[f"gpu{i}_peak_reserved_gb"] = torch.cuda.max_memory_reserved(i) / 2**30
        try:
            q = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,temperature.gpu,power.draw,memory.used",
                                "--format=csv,noheader,nounits"], capture_output=True, text=True,
                               timeout=10, check=False)
            for line in q.stdout.strip().splitlines():
                vals = [x.strip() for x in line.split(",")]
                if len(vals) == 5:
                    i = vals[0]
                    for key, val in zip(("util_pct", "temp_c", "power_w", "mem_used_mb"), vals[1:]):
                        try: rec[f"gpu{i}_{key}"] = float(val)
                        except ValueError: pass
        except Exception:
            pass
    return rec

class Trainer:
    def __init__(self, model, loss_fn, out_dir, run_id, sync=None, lr=5e-4, weight_decay=1e-4,
                 epochs=120, patience=20, batch_size=64, num_workers=2, amp=True,
                 multi_gpu=True, grad_clip=1.0, min_lr=1e-6, log_every=25,
                 checkpoint_every_steps=50, checkpoint_every_s=300, seed=42,
                 require_dual_gpu=False, run_config=None):
        self.device, self.ngpu, self.gpu_names = pick_device()
        if require_dual_gpu and self.ngpu < 2:
            raise RuntimeError("This training notebook requires Kaggle GPU T4 x2. "
                               "Choose Settings > Accelerator > GPU T4 x2, then restart.")
        self.raw_model = model.to(self.device); self.model = self.raw_model
        if multi_gpu and self.ngpu > 1:
            self.model = nn.DataParallel(self.raw_model)
        self.loss_fn = loss_fn
        self.out = Path(out_dir); self.out.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id; self.sync = sync
        self.epochs = int(epochs); self.patience = int(patience)
        self.bs = int(batch_size); self.nw = int(num_workers); self.seed = int(seed)
        self.amp = bool(amp and self.device.type == "cuda"); self.grad_clip = grad_clip
        self.opt = torch.optim.AdamW(self.raw_model.parameters(), lr=lr, weight_decay=weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.opt, T_max=self.epochs, eta_min=min_lr)
        self.scaler = _grad_scaler(self.device.type, self.amp)
        self.log_every = max(1, int(log_every))
        self.checkpoint_every_steps = max(1, int(checkpoint_every_steps))
        self.checkpoint_every_s = max(30, int(checkpoint_every_s))
        self.config = dict(run_config or {})
        self.config_hash = hashlib.sha256(json.dumps(
            self.config, sort_keys=True, default=str).encode()).hexdigest()
        self.state = {"schema_version": 4, "engine_version": ENGINE_VERSION,
                      "epoch": 0, "active_epoch": 0, "batch_in_epoch": 0,
                      "global_step": 0, "best": float("inf"), "best_epoch": -1,
                      "bad_epochs": 0, "history": [], "partial": {},
                      "run_id": run_id, "done": False, "config_hash": self.config_hash,
                      "created_utc": datetime.now(timezone.utc).isoformat()}
        self._save_lock = threading.Lock(); self._last_checkpoint = time.time()
        self._active = False
        _atomic_json(self.out / "run_config.json", self.config)
        _atomic_json(self.out / "environment.json", {
            "engine_version": ENGINE_VERSION, "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu_count": self.ngpu, "gpu_names": self.gpu_names,
            "amp": self.amp, "python": os.sys.version, "config_hash": self.config_hash})

    @property
    def ckpt(self):
        return self.out / "state.pt"

    def _payload(self):
        return {"model": self.raw_model.state_dict(), "opt": self.opt.state_dict(),
                "sched": self.sched.state_dict(), "scaler": self.scaler.state_dict(),
                "state": self.state, "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "np_rng": np.random.get_state(), "python_rng": random.getstate(),
                "config": self.config, "config_hash": self.config_hash,
                "saved_utc": datetime.now(timezone.utc).isoformat()}

    def save(self, tag="state", reason="checkpoint"):
        with self._save_lock:
            path = self.out / (tag + ".pt"); tmp = path.with_suffix(".pt.tmp")
            torch.save(self._payload(), tmp); os.replace(tmp, path)
            _atomic_json(self.out / "state.json", self.state)
            self._last_checkpoint = time.time()
        if self.sync:
            self.sync.mark_dirty(f"{self.run_id}:{reason}")

    def emergency_checkpoint(self):
        if self._active:
            self.save("state", reason="interrupt-emergency")

    def load(self):
        if not self.ckpt.exists():
            return False
        try:
            d = torch.load(self.ckpt, map_location=self.device, weights_only=False)
            got_hash = d.get("config_hash", d.get("state", {}).get("config_hash"))
            if got_hash and got_hash != self.config_hash:
                raise RuntimeError("checkpoint configuration differs from this run. "
                                   "Use a new RUN_ID or restore the original configuration.")
            self.raw_model.load_state_dict(d["model"], strict=True)
            self.opt.load_state_dict(d["opt"]); self.sched.load_state_dict(d["sched"])
            self.scaler.load_state_dict(d["scaler"]); self.state = d["state"]
            torch.set_rng_state(d["torch_rng"].cpu()); np.random.set_state(d["np_rng"])
            random.setstate(d["python_rng"])
            if torch.cuda.is_available() and d.get("cuda_rng"):
                torch.cuda.set_rng_state_all([x.cpu() for x in d["cuda_rng"]])
            print(f"  resumed {self.run_id}: completed_epoch={self.state['epoch']}, "
                  f"active_epoch={self.state.get('active_epoch')}, "
                  f"completed_batches={self.state.get('batch_in_epoch', 0)}")
            return True
        except Exception as e:
            raise RuntimeError(f"Checkpoint exists but cannot be resumed safely: "
                               f"{type(e).__name__}: {e}") from e

    def _loader(self, ds, shuffle, epoch=0):
        if len(ds) == 0:
            raise RuntimeError("empty dataset -- check the subject split")
        if hasattr(ds, "set_epoch"):
            ds.set_epoch(epoch)
        drop = bool(shuffle) and len(ds) > self.bs
        gen = torch.Generator(); gen.manual_seed(self.seed + int(epoch) * 1000003)
        return DataLoader(ds, batch_size=min(self.bs, max(len(ds), 1)), shuffle=shuffle,
                          generator=gen, num_workers=self.nw,
                          pin_memory=(self.device.type == "cuda"), drop_last=drop,
                          persistent_workers=False)

    def _step(self, batch, train):
        x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
        with _autocast(self.device.type, self.amp):
            pred = self.model(x)
            if isinstance(pred, dict) and "aux" in pred and not train:
                pred = {k: v for k, v in pred.items() if k != "aux"}
            loss, parts = self.loss_fn(pred, y, pk, rr)
        return loss, {k: float(v) for k, v in parts.items()}, pred, y, pk, rr

    def _write_batch(self, rec):
        _append_jsonl(self.out / "batch_metrics.jsonl", rec)

    def _validation(self, val_ds, epoch):
        self.model.eval(); sums = {}; n = 0; Y = []; P = []; PK = []; PH = []; RR = []; RH = []
        t0 = time.time()
        with torch.no_grad():
            for batch in self._loader(val_ds, False, epoch):
                loss, parts, pred, y, pk, rr = self._step(batch, False)
                parts = {"total": float(loss), **parts}
                for k, v in parts.items(): sums[k] = sums.get(k, 0.0) + float(v)
                n += 1; Y.append(y.squeeze(1).float().cpu().numpy())
                P.append(pred["wave"].squeeze(1).float().cpu().numpy())
                PK.append(pk.squeeze(1).float().cpu().numpy())
                if "peak" in pred: PH.append(pred["peak"].squeeze(1).float().cpu().numpy())
                RR.append(rr.squeeze(1).float().cpu().numpy())
                if "rr" in pred: RH.append(pred["rr"].squeeze(1).float().cpu().numpy())
        if n == 0: raise RuntimeError("validation loader yielded zero batches")
        y = np.concatenate(Y); p = np.concatenate(P); pk = np.concatenate(PK)
        rec = _mean_parts(sums, n, "val")
        rec["val_seconds"] = time.time() - t0; rec["val_windows"] = len(y)
        rec.update(val_true_mean=float(y.mean()), val_true_std=float(y.std()),
                   val_true_min=float(y.min()), val_true_max=float(y.max()),
                   val_pred_mean=float(p.mean()), val_pred_std=float(p.std()),
                   val_pred_min=float(p.min()), val_pred_max=float(p.max()),
                   val_pred_bias=float((p-y).mean()))
        try:
            from crvs_metrics import seg_metrics, peak_detection_scores, detect_r_peaks, hrv_from_peaks
            global_m = seg_metrics(y.reshape(-1), p.reshape(-1))
            rec.update({"val_" + k: v for k, v in global_m.items()})
            # Per-window waveform diagnostics are compact and retained for every epoch.
            den_y = np.sqrt(np.sum((y - y.mean(1, keepdims=True)) ** 2, axis=1))
            den_p = np.sqrt(np.sum((p - p.mean(1, keepdims=True)) ** 2, axis=1))
            cc = np.sum((y-y.mean(1, keepdims=True))*(p-p.mean(1, keepdims=True)), axis=1) / (den_y*den_p+1e-12)
            win = {"epoch": np.full(len(y), epoch + 1), "window": np.arange(len(y)),
                   "mae": np.mean(np.abs(y-p), 1), "mse": np.mean((y-p)**2, 1),
                   "cc_temporal": 100*cc}
            try:
                import pandas as pd
                frame = pd.DataFrame(win)
                if hasattr(val_ds, "index") and len(val_ds.index) == len(frame):
                    for col in ("rec_id", "subject", "scenario_canon", "start"):
                        if col in val_ds.index: frame[col] = val_ds.index[col].to_numpy()
                vd = self.out / "validation_windows"; vd.mkdir(exist_ok=True)
                frame.to_parquet(vd / f"epoch_{epoch+1:04d}.parquet", index=False)
            except Exception as e:
                rec["val_window_table_error"] = f"{type(e).__name__}: {e}"
            # Never concatenate different recordings: that fabricates a beat interval at
            # each boundary. Compute peak/HRV metrics per recording, then macro-average.
            record_rows = []
            if hasattr(val_ds, "index") and len(val_ds.index) == len(y):
                ix = val_ds.index.reset_index(drop=True).assign(_row=np.arange(len(y)))
                for rid, grp in ix.groupby("rec_id"):
                    pos = grp.sort_values("start")["_row"].to_numpy()
                    if len(pos) < 2: continue
                    yg = np.concatenate(y[pos]); pg = np.concatenate(p[pos])
                    peak_s = peak_detection_scores(yg, pg)
                    gt_hrv = hrv_from_peaks(detect_r_peaks(yg))
                    pr_hrv = hrv_from_peaks(detect_r_peaks(pg))
                    rr = {"rec_id": rid, "subject": str(grp["subject"].iloc[0]), **peak_s}
                    for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                        rr[f"true_{k}"] = gt_hrv[k]; rr[f"pred_{k}"] = pr_hrv[k]
                        rr[f"abs_error_{k}"] = abs(pr_hrv[k]-gt_hrv[k])
                    record_rows.append(rr)
            if record_rows:
                import pandas as pd
                rdf = pd.DataFrame(record_rows)
                rd = self.out / "validation_recordings"; rd.mkdir(exist_ok=True)
                rdf.to_parquet(rd / f"epoch_{epoch+1:04d}.parquet", index=False)
                for k in ("TP", "FP", "FN", "precision", "recall", "F1", "accuracy",
                          "timing_err_ms_median", "timing_err_ms_iqr", "missed_rate"):
                    rec[f"val_wave_peak_{k}"] = float(rdf[k].mean())
                for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                    rec[f"val_hrv_true_{k}"] = float(rdf[f"true_{k}"].mean())
                    rec[f"val_hrv_pred_{k}"] = float(rdf[f"pred_{k}"].mean())
                    rec[f"val_hrv_abs_error_{k}"] = float(rdf[f"abs_error_{k}"].mean())
        except Exception as e:
            rec["val_signal_metrics_error"] = f"{type(e).__name__}: {e}"
        if PH:
            ph = np.concatenate(PH); prob = 1 / (1 + np.exp(-np.clip(ph, -30, 30)))
            truth = pk >= 0.5; guess = prob >= 0.5
            tp = int(np.sum(truth & guess)); fp = int(np.sum(~truth & guess)); fn = int(np.sum(truth & ~guess))
            prec = tp / max(tp+fp, 1); recall = tp / max(tp+fn, 1)
            rec.update(val_peak_head_TP=tp, val_peak_head_FP=fp, val_peak_head_FN=fn,
                       val_peak_head_precision=prec, val_peak_head_recall=recall,
                       val_peak_head_F1=2*prec*recall/max(prec+recall, 1e-12))
        if RH:
            rr = np.concatenate(RR); rh = np.concatenate(RH); mask = rr > 0
            if np.any(mask):
                rec["val_rr_head_mae_ms"] = float(np.mean(np.abs(rr[mask]-rh[mask]))*1000)
                rec["val_rr_head_rmse_ms"] = float(np.sqrt(np.mean((rr[mask]-rh[mask])**2))*1000)
        return rec

    def _write_epoch(self, rec):
        _append_jsonl(self.out / "epoch_metrics.jsonl", rec)
        # CSV is convenient in Kaggle; JSONL remains the lossless schema-of-record.
        rows = self.state["history"]
        keys = sorted({k for r in rows for k in r})
        tmp = self.out / "epoch_metrics.csv.tmp"
        with open(tmp, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
            for row in rows: w.writerow({k: _jsonable(row.get(k)) for k in keys})
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, self.out / "epoch_metrics.csv")

    def fit(self, train_ds, val_ds):
        start = int(self.state["epoch"])
        if start >= self.epochs:
            print(f"  {self.run_id} already complete at epoch {start}/{self.epochs}")
            return self.state
        if self.state.get("done"):
            self.state["done"] = False
        self._active = True
        if self.sync: self.sync.set_before_final_flush(self.emergency_checkpoint)
        run_t0 = time.time()
        try:
            for ep in range(start, self.epochs):
                if torch.cuda.is_available():
                    for gpu_i in range(torch.cuda.device_count()):
                        torch.cuda.reset_peak_memory_stats(gpu_i)
                tl = self._loader(train_ds, True, ep)
                resume_batch = int(self.state.get("batch_in_epoch", 0)) if int(self.state.get("active_epoch", ep)) == ep else 0
                partial = self.state.get("partial", {}) if resume_batch else {}
                sums = {k: float(v) for k, v in partial.get("sums", {}).items()}
                n = int(partial.get("n", 0)); samples = int(partial.get("samples", 0))
                grad_sum = float(partial.get("grad_sum", 0)); clip_events = int(partial.get("clip_events", 0))
                epoch_t0 = time.time(); data_t = 0.0; compute_t = 0.0; last_end = time.time()
                self.state.update(active_epoch=ep, batch_in_epoch=resume_batch, done=False)
                self.model.train()
                for i, batch in enumerate(tl):
                    data_t += time.time() - last_end
                    if i < resume_batch:
                        last_end = time.time(); continue
                    step_t0 = time.time(); self.opt.zero_grad(set_to_none=True)
                    loss, parts, _, _, _, _ = self._step(batch, True)
                    if not torch.isfinite(loss):
                        self.save("state", reason="non-finite-loss")
                        raise FloatingPointError(f"non-finite loss at epoch {ep+1}, batch {i+1}")
                    self.scaler.scale(loss).backward(); self.scaler.unscale_(self.opt)
                    grad = float(torch.nn.utils.clip_grad_norm_(
                        self.raw_model.parameters(), self.grad_clip or float("inf")))
                    if self.grad_clip and grad > self.grad_clip: clip_events += 1
                    self.scaler.step(self.opt); self.scaler.update()
                    compute_t += time.time() - step_t0
                    values = {"total": float(loss.detach()), **parts}
                    for k, v in values.items(): sums[k] = sums.get(k, 0.0) + float(v)
                    n += 1; samples += int(batch[0].shape[0]); grad_sum += grad
                    self.state["global_step"] = int(self.state.get("global_step", 0)) + 1
                    self.state["batch_in_epoch"] = i + 1
                    self.state["partial"] = {"sums": sums, "n": n, "samples": samples,
                                             "grad_sum": grad_sum, "clip_events": clip_events}
                    if (i + 1) % self.log_every == 0 or i + 1 == len(tl):
                        brec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                                "epoch": ep+1, "batch": i+1, "batches": len(tl),
                                "global_step": self.state["global_step"], "loss": float(loss),
                                "grad_norm": grad, "lr": self.opt.param_groups[0]["lr"],
                                "amp_scale": float(self.scaler.get_scale()),
                                "windows_per_s": int(batch[0].shape[0])/max(time.time()-step_t0, 1e-9)}
                        brec.update({"loss_"+k: v for k, v in parts.items()}); self._write_batch(brec)
                    due_step = self.state["global_step"] % self.checkpoint_every_steps == 0
                    due_time = time.time() - self._last_checkpoint >= self.checkpoint_every_s
                    if due_step or due_time:
                        self.save("state", reason="mid-epoch")
                    last_end = time.time()
                if n == 0: raise RuntimeError("training loader yielded zero batches")
                self.sched.step(); val = self._validation(val_ds, ep)
                rec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                       "epoch": ep+1, "epochs_planned": self.epochs,
                       "global_step": self.state["global_step"], "train_batches": n,
                       "train_windows": samples, "train_grad_norm_mean": grad_sum/max(n,1),
                       "train_grad_clip_events": clip_events,
                       "train_data_seconds": data_t, "train_compute_seconds": compute_t,
                       "train_windows_per_s": samples/max(compute_t, 1e-9),
                       "epoch_seconds": time.time()-epoch_t0,
                       "elapsed_seconds": time.time()-run_t0,
                       "lr": self.opt.param_groups[0]["lr"],
                       "amp_scale": float(self.scaler.get_scale())}
                rec.update(_mean_parts(sums, n, "train")); rec.update(val); rec.update(_system_stats(self.out))
                with torch.no_grad():
                    rec["model_parameter_l2"] = math.sqrt(sum(
                        float(torch.sum(p.detach().float() ** 2)) for p in self.raw_model.parameters()))
                va = float(rec["val_total"])
                improved = va < float(self.state["best"]) - 1e-6
                if improved:
                    self.state["best"] = va; self.state["best_epoch"] = ep+1
                    self.state["bad_epochs"] = 0
                else:
                    self.state["bad_epochs"] = int(self.state.get("bad_epochs", 0)) + 1
                rec.update(improved=bool(improved), best_val=float(self.state["best"]),
                           best_epoch=int(self.state["best_epoch"]),
                           bad_epochs=int(self.state["bad_epochs"]))
                self.state["epoch"] = ep+1; self.state["active_epoch"] = ep+1
                self.state["batch_in_epoch"] = 0; self.state["partial"] = {}
                self.state["history"].append(rec)
                if improved: self.save("best", reason="new-best-local")
                self.save("state", reason="epoch-complete"); self._write_epoch(rec)
                if self.sync:
                    self.sync.log("epoch", run_id=self.run_id, epoch=ep+1,
                                  train_total=rec.get("train_total"), val_total=va,
                                  cc_t=rec.get("val_CC_temporal"), cc_s=rec.get("val_CC_spectral"),
                                  hr_mae_bpm=rec.get("val_hrv_abs_error_mean_hr_bpm"), improved=improved)
                eta = rec["epoch_seconds"] * max(self.epochs-ep-1, 0) / 3600
                print(f"  ep {ep+1:>3}/{self.epochs} train {rec['train_total']:.5f} "
                      f"val {va:.5f} CCt {rec.get('val_CC_temporal', float('nan')):.1f} "
                      f"CCs {rec.get('val_CC_spectral', float('nan')):.1f} "
                      f"{'*' if improved else ''} {rec['epoch_seconds']:.0f}s ETA {eta:.1f}h")
                if int(self.state["bad_epochs"]) >= self.patience:
                    self.state["stop_reason"] = "early_stopping"; break
            self.state["done"] = True
            self.state["finished_utc"] = datetime.now(timezone.utc).isoformat()
            self.save("state", reason="run-complete")
            if self.sync: self.sync.log("training_complete", run_id=self.run_id)
            return self.state
        except KeyboardInterrupt:
            # SIGINT normally reaches HFSync first: its final hook already saved and pushed.
            # Avoid a duplicate commit; a directly-raised KeyboardInterrupt still takes this path.
            if time.time() - self._last_checkpoint > 2:
                self.save("state", reason="keyboard-interrupt")
            if self.sync:
                if not self.sync.recently_pushed(5):
                    self.sync.flush(final=True, force=True,
                                    msg=f"{self.run_id} stopped by user", run_final_hook=False)
            raise
        except Exception as e:
            self.state["last_error"] = f"{type(e).__name__}: {e}"
            self.save("state", reason="training-error")
            if self.sync:
                self.sync.log("training_error", run_id=self.run_id, error=self.state["last_error"])
                self.sync.flush(final=True, force=True,
                                msg=f"{self.run_id} error checkpoint", run_final_hook=False)
            raise
        finally:
            self._active = False
            if self.sync: self.sync.set_before_final_flush(None)

    @torch.no_grad()
    def predict(self, ds, max_keep=None):
        bp = self.out / "best.pt"
        if bp.exists():
            d = torch.load(bp, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(d["model"], strict=True)
        self.model.eval(); Y = []; P = []
        for batch in self._loader(ds, False, 0):
            x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
            with _autocast(self.device.type, self.amp): out = self.model(x)
            Y.append(y.squeeze(1).float().cpu().numpy())
            P.append(out["wave"].squeeze(1).float().cpu().numpy())
        Y = np.concatenate(Y); P = np.concatenate(P)
        if max_keep is not None: return Y[:max_keep], P[:max_keep]
        return Y, P
""",
}
for nm, src in MODULES.items():
    (WORK / nm).write_text(src)
    print(f"  {nm:<20} {len(src):>7,} chars")
import hashlib
MODULE_HASHES = {nm: hashlib.sha256(src.encode()).hexdigest() for nm, src in MODULES.items()}
(WORK / "library_hashes.json").write_text(json.dumps(MODULE_HASHES, indent=2))

# Writing a .py and importing it is NOT idempotent inside one kernel: Python caches the
# module object in sys.modules, so re-running this cell after updating the notebook keeps
# the OLD code. That is exactly how a stale .npz loader survived a rebuilt notebook and
# produced a FileNotFoundError deep inside a DataLoader worker. Purge and re-import.
import importlib
for nm in MODULES:
    sys.modules.pop(nm[:-3], None)
importlib.invalidate_caches()

import crvs_data
REQUIRED_LIB = 4
if getattr(crvs_data, "LIB_VERSION", 0) < REQUIRED_LIB:
    raise RuntimeError(
        f"\n{'='*74}\n  Stale crvs_data: version "
        f"{getattr(crvs_data, 'LIB_VERSION', 'missing')}, need >= {REQUIRED_LIB}."
        f"\n  Loaded from {getattr(crvs_data, '__file__', '?')}"
        f"\n  Restart the kernel (Run -> Restart & Run All) and try again.\n{'='*74}")
print(f"\nlibrary written to {WORK}  |  crvs_data v{crvs_data.LIB_VERSION} loaded from "
      f"{crvs_data.__file__}")

  crvs_sync.py          10,952 chars
  crvs_data.py           8,843 chars
  crvs_metrics.py        5,224 chars
  crvs_models.py        10,854 chars
  crvs_losses.py         3,783 chars
  crvs_engine.py        26,563 chars

library written to /kaggle/working/nb03  |  crvs_data v4 loaded from /kaggle/working/nb03/crvs_data.py


In [4]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found. Add-ons -> Secrets -> HF_TOKEN (write) -> attach.\n" + "="*74)

# Verify the one-time reset before constructing HFSync. If the guard fails, no uploader,
# signal handler or atexit hook exists yet, so this notebook cannot modify the old repo.
if CFG["REQUIRE_CLEAN_RESET"]:
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(repo_id=CFG["DST_REPO"], repo_type="model",
                        filename=CFG["RESET_RECEIPT"], token=HF_TOKEN,
                        local_dir=str(WORK))
    except Exception as e:
        raise RuntimeError(
            "\n" + "="*74 +
            "\n  One-time baseline cleanup receipt is missing."
            "\n  Run 02b_cleanup_baselines_hf_once.ipynb first, then start a NEW"
            "\n  Kaggle session and Run All here. This guard prevents old smoke-test"
            "\n  checkpoints from being mixed with the canonical baseline queue."
            f"\n  Hub check: {type(e).__name__}: {str(e)[:180]}"
            "\n" + "="*74) from e

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["DST_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="model",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"],
              max_upload_calls_hour=CFG["HF_MAX_UPLOADS_HOUR"])
print("\nresults repo:", sync.url, "(public)")

_major_state = {"name": None}
def MAJOR(nm, _state=_major_state):
    _state["name"] = str(nm)
def _nb03_post_run_cell(r=None, _state=_major_state, _sync=sync):
    nm = _state.pop("name", None)
    if nm is not None:
        _sync.stage_done(nm)
_nb03_post_run_cell._crvs_hook_id = "nb03-v3"
try:
    _ip = get_ipython()
    for _old in list(_ip.events.callbacks.get("post_run_cell", [])):
        if str(getattr(_old, "_crvs_hook_id", "")).startswith("nb03-"):
            try: _ip.events.unregister("post_run_cell", _old)
            except Exception: pass
    _ip.events.register("post_run_cell", _nb03_post_run_cell)
    print("post-run-cell push hook registered")
except Exception as e:
    print("hook unavailable:", e)

# Resume: pull back the small artefacts (state, summaries, metrics) but NOT the weights --
# finished runs never need their checkpoints re-downloaded, only their summary.json.
sync.pull(allow_patterns=["*.json", "*.jsonl", "*.csv", "*.md", "runs/**/summary.json",
                          "runs/**/state.json", "runs/B_rva__*/preds_sample.npz",
                          "runs/**/epoch_metrics.csv",
                          "results/*", "cleanup_receipts/*.json"])
STATE = sync.load_state({"completed": [], "sessions": 0, "version": 2})
STATE["version"] = 3
STATE["protocol_id"] = CFG["PROTOCOL_ID"]
STATE["sessions"] = STATE.get("sessions", 0) + 1
sync.save_state(STATE)
print(f"session #{STATE['sessions']}  |  {len(STATE['completed'])} run(s) already complete")
MAJOR("00_setup")

HF_TOKEN loaded from Kaggle Secrets.


(…)selines-v2-canonical-reset-20260902.json: 0.00B [00:00, ?B/s]

  [sync_started] repo=Shanmuk4622/cardiomamba-baselines-v2 private=False interval_s=1800 sync_version=2

results repo: https://huggingface.co/Shanmuk4622/cardiomamba-baselines-v2 (public)
post-run-cell push hook registered


Fetching ... files: 0it [00:00, ?it/s]

  [resume_pull_ok] path=/kaggle/working/nb03 patterns=['*.json', '*.jsonl', '*.csv', '*.md', 'runs/**/summary.json', 'runs/**/state.json', 'runs/B_rva__*/preds_sample.npz', 'runs/**/epoch_metrics.csv', 'results/*', 'cleanup_receipts/*.json']
session #11  |  75 run(s) already complete
  [stage_done] stage=00_setup


---
# 3 · Fetch the corpus

Recordings are pulled to **scratch**, not to `/kaggle/working`, so the ~400 MB of data never eats
into the 20 GB output budget that the checkpoints need.

In [5]:
from huggingface_hub import snapshot_download

# Prefer NB02's saved Kaggle notebook output: it mounts instantly and costs no working disk.
DATA = None
input_root = Path("/kaggle/input")
if input_root.exists():
    for candidate in input_root.rglob("windows.parquet"):
        if (candidate.parent / "recordings").exists() and (candidate.parent / "norm_stats.json").exists():
            DATA = candidate.parent; break
if DATA is not None:
    print("using attached Kaggle NB02 output:", DATA)
else:
    DATA = SCRATCH / "corpus"; t0 = time.time()
    print("NB02 output not attached; falling back to Hugging Face download.")
    snapshot_download(CFG["SRC_REPO"], repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA),
                      allow_patterns=["recordings/*.npy", "recordings/*.json",
                                      "recordings/*.npz", "windows.parquet", "recordings.csv",
                                      "norm_stats.json", "experiments.json"], max_workers=4)
    print(f"corpus downloaded in {time.time()-t0:.0f}s")

W = pd.read_parquet(DATA / "windows.parquet")
DATA_HASH = hashlib.sha256((DATA / "windows.parquet").read_bytes()).hexdigest()
RECS = pd.read_csv(DATA / "recordings.csv")
NORM = json.loads((DATA / "norm_stats.json").read_text())
EXPINFO = json.loads((DATA / "experiments.json").read_text())
EXPERIMENTS = EXPINFO["experiments"]
REC_DIR = DATA / "recordings"

# Fail here, loudly, rather than inside a DataLoader worker 20 minutes into a run.
_npy = {p.stem for p in (REC_DIR).glob("*.npy")}
_npz = {p.stem for p in (REC_DIR).glob("*.npz")}
_have = _npy | _npz
print(f"recording files: {len(_npy)} .npy (fast path) + {len(_npz)} .npz (legacy, ~85x "
      f"slower per window)")
if not _have:
    raise RuntimeError(
        "\n" + "=" * 74 +
        "\n  No recording files downloaded."
        "\n  Check that NB02 finished and pushed, and that SRC_REPO matches its DST_REPO."
        "\n" + "=" * 74)
_missing = sorted(set(RECS["rec_id"]) - _have)
if _missing:
    print(f"WARNING: {len(_missing)} recording(s) in recordings.csv have no file: "
          f"{_missing[:5]}{' ...' if len(_missing) > 5 else ''}")
    W = W[~W["rec_id"].isin(_missing)].reset_index(drop=True)
    RECS = RECS[~RECS["rec_id"].isin(_missing)].reset_index(drop=True)
    print(f"         dropped their windows; {len(W):,} remain")

print(f"windows      : {len(W):,}   ({int(W['no_overlap'].sum()):,} non-overlapping)")
print(f"recordings   : {len(RECS)}   subjects: {RECS['subject'].nunique()}")
print(f"norm sets    : {len(NORM)}")
print(f"channels     : {EXPINFO['channels']}")
print(f"using        : {CFG['CHANNELS']}  <- the baseline's single-channel input")
print("\nwindows per experiment:")
for e, sc in EXPERIMENTS.items():
    sub = W[W["scenario_canon"].isin(sc)]
    print(f"  {e:<12} {len(sub):>8,} windows  {sub['subject'].nunique():>3} subjects  {sc}")

NB02 output not attached; falling back to Hugging Face download.


Fetching 272 files:   0%|          | 0/272 [00:00<?, ?it/s]

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

corpus downloaded in 16s
recording files: 134 .npy (fast path) + 0 .npz (legacy, ~85x slower per window)
windows      : 20,757   (10,411 non-overlapping)
recordings   : 134   subjects: 30
norm sets    : 60
channels     : ['I', 'Q', 'phi', 'dy', 'vel', 'acc', 'amp', 'cardiac']
using        : ['dy']  <- the baseline's single-channel input

windows per experiment:
  A_resting       4,609 windows   30 subjects  ['Resting']
  A_valsalva      6,791 windows   27 subjects  ['Valsalva']
  A_apnea         1,115 windows   24 subjects  ['Apnea']
  B_rva          12,515 windows   30 subjects  ['Resting', 'Valsalva', 'Apnea']
  C_all5         20,757 windows   30 subjects  ['Resting', 'Valsalva', 'Apnea', 'Tilt-up', 'Tilt-down']


---
# 4 · Model smoke test

Before spending GPU hours, prove each network actually runs: correct output shape, finite values,
gradients that flow, and a parameter count we can report. The baseline paper reports **no**
parameter or FLOP budget for any of its models — we will, for all of them.

In [6]:
from crvs_models import build_baseline, count_params
from crvs_losses import MSEOnly, CompositeLoss
from crvs_engine import Trainer, seed_all, pick_device
import crvs_engine

# ---------------------------------------------------------------------------
# Long-session host-RAM safety patch
# ---------------------------------------------------------------------------
# This is deliberately a runtime/resource patch, not a change to the embedded model, loss,
# data, or optimiser libraries.  Existing checkpoints therefore keep the same config hash
# and resume safely.  It addresses the hard Kaggle restart seen after RAM climbed across
# epochs/runs, while preserving the exact training mathematics and validation outputs.
MEMORY_RUNTIME_VERSION = "nb03-process-isolated-v1"

def _linux_memory_gb():
    """Current process RSS and machine available RAM (not the monotonic peak RSS)."""
    out = {"process_rss_gb": float("nan"), "host_ram_available_gb": float("nan")}
    try:
        status = Path("/proc/self/status").read_text()
        for line in status.splitlines():
            if line.startswith("VmRSS:"):
                out["process_rss_gb"] = float(line.split()[1]) / 2**20
                break
    except Exception:
        pass
    try:
        meminfo = Path("/proc/meminfo").read_text()
        for line in meminfo.splitlines():
            if line.startswith("MemAvailable:"):
                out["host_ram_available_gb"] = float(line.split()[1]) / 2**20
                break
    except Exception:
        pass
    return out

def release_runtime_memory(label=None, aggressive=False):
    """Return unused Python, Arrow and glibc pages to the OS; safe on non-Linux too."""
    if aggressive and torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    gc.collect()
    try:
        import pyarrow as pa
        pa.default_memory_pool().release_unused()
    except Exception:
        pass
    for _ in range(2 if aggressive else 1):
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass
        if aggressive:
            gc.collect()
    stats = _linux_memory_gb()
    if label:
        print(f"  RAM after {label}: {stats['process_rss_gb']:.2f} GB RSS | "
              f"{stats['host_ram_available_gb']:.2f} GB host available")
    return stats

def _migrate_loss_metric_names(record):
    # Evaluation MSE is `val_MSE`; the lowercase field is the validation loss component.
    # Giving the latter an explicit name makes the CSV usable by case-insensitive readers.
    for old, new in (("train_mse", "train_loss_mse"),
                     ("val_mse", "val_loss_mse")):
        if old in record:
            record.setdefault(new, record[old])
            record.pop(old, None)

if not getattr(Trainer, "_nb03_host_ram_patch", False):
    _base_validation = Trainer._validation
    _base_write_epoch = Trainer._write_epoch
    _base_load = Trainer.load
    _base_loader = Trainer._loader
    _base_payload = Trainer._payload
    _base_save = Trainer.save
    _base_system_stats = crvs_engine._system_stats

    def _cpu_checkpoint_tree(value, cache, path=()):
        # torch.save on live CUDA tensors retained approximately one checkpoint-sized host
        # transfer buffer per epoch (146 MB/epoch for UNet in the observed Hub telemetry).
        # Copy into one reusable pageable-CPU buffer per tensor, then serialize those buffers.
        if torch.is_tensor(value):
            src = value.detach()
            buf = cache.get(path)
            if (buf is None or tuple(buf.shape) != tuple(src.shape) or
                    buf.dtype != src.dtype or buf.layout != src.layout):
                buf = torch.empty_like(src, device="cpu", memory_format=torch.preserve_format)
                cache[path] = buf
            buf.copy_(src, non_blocking=False)
            return buf
        if isinstance(value, dict):
            return {k: _cpu_checkpoint_tree(v, cache, path + (("dict", repr(k)),))
                    for k, v in value.items()}
        if isinstance(value, list):
            return [_cpu_checkpoint_tree(v, cache, path + (("list", i),))
                    for i, v in enumerate(value)]
        if isinstance(value, tuple):
            return tuple(_cpu_checkpoint_tree(v, cache, path + (("tuple", i),))
                         for i, v in enumerate(value))
        return value

    def _bounded_payload(self):
        cache = getattr(self, "_nb03_cpu_checkpoint_buffers", None)
        if cache is None:
            cache = {}
            self._nb03_cpu_checkpoint_buffers = cache
        return _cpu_checkpoint_tree(_base_payload(self), cache)

    def _bounded_save(self, *args, **kwargs):
        try:
            return _base_save(self, *args, **kwargs)
        finally:
            # The reusable buffers stay allocated; transient serialization pages do not.
            release_runtime_memory()

    def _bounded_loader(self, *args, **kwargs):
        loader = _base_loader(self, *args, **kwargs)
        # The base engine enables CUDA pinning.  It is useful for large image batches but
        # retained pinned pages were harmful in this multi-hour, memory-mapped 1-D workload.
        loader.pin_memory = bool(CFG["PIN_MEMORY"])
        return loader

    def _bounded_validation(self, *args, **kwargs):
        try:
            return _base_validation(self, *args, **kwargs)
        finally:
            # The original validation result contains scalars only.  Once it returns, its
            # large waveform/SciPy/Arrow temporaries can be released before the next epoch.
            stats = release_runtime_memory()
            self._nb03_rss_after_validation = stats.get("process_rss_gb", float("nan"))
            if (np.isfinite(self._nb03_rss_after_validation) and
                    self._nb03_rss_after_validation >= CFG["HOST_RAM_WARN_GB"] and
                    not getattr(self, "_nb03_ram_warning_emitted", False)):
                self._nb03_ram_warning_emitted = True
                print(f"  RAM warning: {self._nb03_rss_after_validation:.2f} GB RSS; "
                      "continuing the queue (no automatic RAM stop).")
                if self.sync:
                    self.sync.log("host_memory_warning", run_id=self.run_id,
                                  rss_gb=self._nb03_rss_after_validation)

    def _portable_write_epoch(self, rec):
        for old_rec in self.state.get("history", []):
            _migrate_loss_metric_names(old_rec)
        _migrate_loss_metric_names(rec)
        _base_write_epoch(self, rec)

    def _portable_load(self, *args, **kwargs):
        loaded = _base_load(self, *args, **kwargs)
        for old_rec in self.state.get("history", []):
            _migrate_loss_metric_names(old_rec)
        # A RAM guard is a clean pause, not a model/training failure.  Remove the stale
        # diagnostic once its exact checkpoint has been restored for continuation.
        if self.state.get("stop_reason") == "host_memory_guard":
            self.state.pop("stop_reason", None)
            if str(self.state.get("last_error", "")).startswith("HostMemoryGuard:"):
                self.state.pop("last_error", None)
        return loaded

    def _memory_stats(out_dir):
        rec = _base_system_stats(out_dir)
        rec.update(_linux_memory_gb())
        rec["memory_runtime_version"] = MEMORY_RUNTIME_VERSION
        return rec

    Trainer._validation = _bounded_validation
    Trainer._loader = _bounded_loader
    Trainer._payload = _bounded_payload
    Trainer.save = _bounded_save
    Trainer._write_epoch = _portable_write_epoch
    Trainer.load = _portable_load
    Trainer._nb03_host_ram_patch = True
    crvs_engine._system_stats = _memory_stats

def close_dataset(ds):
    """Explicitly close cached npy/npz handles when a fold finishes or fails."""
    base = getattr(ds, "base", ds)
    cache = getattr(base, "_cache", None)
    if not isinstance(cache, dict):
        return
    for rec in list(cache.values()):
        data = getattr(rec, "data", None)
        try:
            if hasattr(data, "close"):
                data.close()
            mmap = getattr(data, "_mmap", None)
            if mmap is not None and not getattr(mmap, "closed", False):
                mmap.close()
        except Exception:
            pass
    cache.clear()

# Repair already-downloaded CSVs from runs created before the loss-field naming fix.
_csv_repairs = 0
for _metric_csv in (WORK / "runs").glob("*/epoch_metrics.csv"):
    try:
        _df = pd.read_csv(_metric_csv)
        _rename = {k: v for k, v in (("train_mse", "train_loss_mse"),
                                     ("val_mse", "val_loss_mse"))
                   if k in _df.columns and v not in _df.columns}
        _dropped_duplicate = False
        for _old, _new in (("train_mse", "train_loss_mse"),
                           ("val_mse", "val_loss_mse")):
            if _old in _df.columns and _new in _df.columns:
                _df.drop(columns=[_old], inplace=True)
                _dropped_duplicate = True
                _csv_repairs += 1
        if _rename:
            _df.rename(columns=_rename).to_csv(_metric_csv, index=False)
            _csv_repairs += 1
        elif _dropped_duplicate:
            _df.to_csv(_metric_csv, index=False)
    except Exception as _csv_error:
        print(f"  metric CSV repair skipped for {_metric_csv.parent.name}: {_csv_error}")

_ram0 = release_runtime_memory("runtime patch")
print(f"memory policy: {MEMORY_RUNTIME_VERSION} | "
      f"{CFG['EPOCHS_PER_PROCESS']} epochs/child process | workers={CFG['WORKERS']} | "
      f"pin_memory={CFG['PIN_MEMORY']} | repaired CSVs={_csv_repairs}")

class BaselineOutputConvention(torch.nn.Module):
    def __init__(self, base, target_01=False):
        super().__init__(); self.base = base; self.target_01 = bool(target_01)
    def forward(self, x):
        out = self.base(x)
        if self.target_01:
            out["wave"] = (out["wave"] + 1.0) * 0.5
            if "aux" in out:
                out["aux"] = [torch.sigmoid(a) for a in out["aux"]]
        return out

def make_baseline(name):
    base = build_baseline(name, in_ch=len(CFG["CHANNELS"]), out_ch=1,
                          base=CFG["BASE"], levels=CFG["LEVELS"])
    return BaselineOutputConvention(base, target_01=CFG["TARGET_01"])

seed_all(CFG["SEED"])
dev, ngpu, names = pick_device()
print(f"device: {dev}  gpus: {ngpu} {names}\n")

C_IN = len(CFG["CHANNELS"])
x = torch.randn(4, C_IN, 1024, device=dev)
y = (torch.rand(4, 1, 1024, device=dev) if CFG["TARGET_01"] else
     torch.randn(4, 1, 1024, device=dev).clamp(-1, 1))
rows = []
print(f"{'model':<20}{'params':>12}{'MB':>8}{'out shape':>18}{'fwd ms':>9}  grad")
print("-" * 78)
for nm in CFG["MODELS"]:
    m = make_baseline(nm).to(dev)
    m.train()
    t0 = time.time()
    out = m(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) * 1000
    loss = torch.nn.functional.mse_loss(out["wave"], y)
    if "aux" in out:
        loss = loss + sum(torch.nn.functional.mse_loss(a, y) for a in out["aux"]) * 0.1
    loss.backward()
    gn = sum(float(p.grad.norm()) for p in m.parameters() if p.grad is not None)
    p = count_params(m)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            m(x[:1])
        gflops = float(fc.get_total_flops()) / 1e9
    except Exception:
        gflops = float("nan")
    ok = (out["wave"].shape == y.shape and torch.isfinite(out["wave"]).all() and gn > 0)
    rows.append({"model": nm, "params": p, "mb": p * 4 / 2**20,
                 "gflops_per_window": gflops, "forward_ms_batch4": dt, "ok": bool(ok)})
    print(f"{nm:<20}{p:>12,}{p*4/2**20:>8.1f}{str(tuple(out['wave'].shape)):>18}"
          f"{dt:>9.1f}  {'OK' if ok else 'FAIL'}")
    del m, out, loss
    release_runtime_memory()
    if dev.type == "cuda":
        torch.cuda.empty_cache()
assert all(r["ok"] for r in rows), "a baseline failed its smoke test"
pd.DataFrame(rows).to_csv(WORK / "results" / "model_budget.csv", index=False)
BUDGET_BY_MODEL = pd.DataFrame(rows).set_index("model").to_dict("index")
del x, y
release_runtime_memory("smoke tensors", aggressive=True)
print("\nall four baselines forward, backward and produce finite output.")
MAJOR("01_smoke")

  RAM after runtime patch: 0.78 GB RSS | 29.68 GB host available
memory policy: nb03-process-isolated-v1 | 5 epochs/child process | workers=0 | pin_memory=False | repaired CSVs=0
device: cuda  gpus: 2 ['Tesla T4', 'Tesla T4']

model                     params      MB         out shape   fwd ms  grad
------------------------------------------------------------------------------
fpn                    2,237,313     8.5      (4, 1, 1024)    838.5  OK
unet                  12,210,881    46.6      (4, 1, 1024)     43.6  OK
linknet                2,775,361    10.6      (4, 1, 1024)     27.7  OK
multireslinknet        9,218,556    35.2      (4, 1, 1024)     60.6  OK
  RAM after smoke tensors: 1.41 GB RSS | 29.18 GB host available

all four baselines forward, backward and produce finite output.
  [stage_done] stage=01_smoke


---
# 5 · The run queue

Each entry is one (experiment, model, fold). Completed runs are read from HF state and skipped,
so re-running the notebook in a fresh session simply continues.

The IDs below are the only baseline run IDs accepted by the result tables. Trial folders and
artefacts from any other protocol are ignored even if they exist remotely.

In [7]:
QUEUE = []
for exp in CFG["EXPERIMENTS"]:
    for mdl in CFG["MODELS"]:
        for f in range(CFG["N_FOLDS"]):
            QUEUE.append({"run_id": f"{exp}__{mdl}__f{f}", "exp": exp,
                          "model": mdl, "fold": f})

queue_ids = {q["run_id"] for q in QUEUE}
done_all = set(STATE.get("completed", []))
# A matching summary is the completion authority. State alone is only a resume hint: if a
# session died after marking state but before uploading its summary, the run must be revisited
# (Trainer.load then resumes or evaluates it without blindly retraining).
summary_done = set()
for _p in (WORK / "runs").glob("*/summary.json"):
    try:
        _s = json.loads(_p.read_text())
        if (_s.get("run_id") in queue_ids and
                _s.get("protocol_id") == CFG["PROTOCOL_ID"]):
            summary_done.add(_s["run_id"])
    except Exception:
        pass
state_only = (done_all & queue_ids) - summary_done
if state_only:
    print(f"repairing {len(state_only)} completion marker(s) without a canonical summary")
done = summary_done
done_all = (done_all - queue_ids) | done
STATE["completed"] = sorted(done_all)
sync.save_state(STATE)
todo = [q for q in QUEUE if q["run_id"] not in done]
print(f"queue: {len(QUEUE)} run(s) total | {len(done)} done | {len(todo)} remaining")
print(f"epochs per run: {CFG['EPOCHS']}   time budget: {CFG['TIME_BUDGET_H']} h")
print("protocol:", CFG["PROTOCOL_ID"])
print("\nnext up:")
for q in todo[:8]:
    print("   ", q["run_id"])
if len(todo) > 8:
    print(f"    ... and {len(todo)-8} more")

queue: 80 run(s) total | 75 done | 5 remaining
epochs per run: 120   time budget: 11.25 h
protocol: baseline-v3-subjectwise-5fold-dy-mse-target01

next up:
    A_apnea__multireslinknet__f0
    A_apnea__multireslinknet__f1
    A_apnea__multireslinknet__f2
    A_apnea__multireslinknet__f3
    A_apnea__multireslinknet__f4


In [8]:
from crvs_data import WindowDataset, WINDOW, FS
from crvs_metrics import seg_metrics, detect_r_peaks, hrv_from_peaks, peak_detection_scores

class TargetConventionDataset:
    # The processed corpus stores ECG in [-1, 1]. The published baseline trained in [0, 1].
    # Keep the conversion at the dataset boundary so training, validation, testing and saved
    # predictions all use one declared convention.
    def __init__(self, base, target_01=False):
        self.base = base
        self.target_01 = bool(target_01)
        self.index = base.index
    def __len__(self):
        return len(self.base)
    def set_epoch(self, epoch):
        self.base.set_epoch(epoch)
    def __getitem__(self, i):
        x, y, pk, rr = self.base[i]
        if self.target_01:
            y = (y + 1.0) * 0.5
        return x, y, pk, rr

def split_for(exp, fold, n_folds=None):
    n_folds = n_folds or CFG["N_FOLDS"]
    sub = W[W["scenario_canon"].isin(EXPERIMENTS[exp])]
    te_g, va_g = fold % n_folds, (fold + 1) % n_folds
    tr = sub[~sub["fold_group"].isin([te_g, va_g])]
    va = sub[(sub["fold_group"] == va_g) & sub["no_overlap"]]
    te = sub[(sub["fold_group"] == te_g) & sub["no_overlap"]]
    assert not (set(tr["subject"]) & set(te["subject"])), "SUBJECT LEAK"
    return tr, va, te

def make_datasets(exp, fold):
    tr, va, te = split_for(exp, fold)
    norm = NORM.get(f"{exp}|{fold}")
    if norm is None:
        raise RuntimeError(f"no normalisation stats for {exp}|{fold} -- re-run NB02")
    idx = [EXPINFO["channels"].index(c) for c in CFG["CHANNELS"]]
    sub_norm = {"mean": [norm["mean"][i] for i in idx],
                "std":  [norm["std"][i] for i in idx]}
    def mk(d, aug):
        base = WindowDataset(REC_DIR, d, sub_norm, CFG["CHANNELS"], augment=aug,
                             seed=CFG["SEED"] + fold)
        return TargetConventionDataset(base, target_01=CFG["TARGET_01"])
    return mk(tr, True), mk(va, False), mk(te, False), (tr, va, te)

def evaluate(Y, P, index, out_dir):
    # Per-window metrics, then per-subject and overall aggregates. Saving per-window rows
    # lets NB05 run the statistics without ever re-running a model.
    rows = []
    idx_frame = index.reset_index(drop=True)
    subs = idx_frame["subject"].to_numpy()
    scen = idx_frame["scenario_canon"].to_numpy()
    for i in range(len(Y)):
        m = seg_metrics(Y[i], P[i], FS)
        m["subject"] = subs[i] if i < len(subs) else "?"
        m["scenario"] = scen[i] if i < len(scen) else "?"
        rows.append(m)
    dfw = pd.DataFrame(rows)
    dfw.to_parquet(out_dir / "metrics_windows.parquet", index=False)
    num = [c for c in dfw.columns if dfw[c].dtype.kind in "fi"]
    agg = {c: float(dfw[c].mean()) for c in num}
    agg.update({c + "_std": float(dfw[c].std()) for c in num})
    # HR/HRV is computed recording-by-recording in chronological window order. Joining
    # different recordings/scenarios would invent an RR interval at every boundary.
    hr_rows = []
    for rid, grp in idx_frame.assign(_row=np.arange(len(idx_frame))).groupby("rec_id"):
        grp = grp.sort_values("start")
        pos = grp["_row"].to_numpy()
        if len(pos) < 2:
            continue
        yg = np.concatenate(Y[pos]); yp = np.concatenate(P[pos])
        g = hrv_from_peaks(detect_r_peaks(yg, FS), FS)
        p = hrv_from_peaks(detect_r_peaks(yp, FS), FS)
        pk = peak_detection_scores(yg, yp, FS)
        hr_rows.append({"rec_id": rid, "subject": str(grp["subject"].iloc[0]),
                        "scenario": str(grp["scenario_canon"].iloc[0]),
                        **{f"gt_{k}": v for k, v in g.items()},
                        **{f"pr_{k}": v for k, v in p.items()}, **pk})
    dfh = pd.DataFrame(hr_rows)
    if len(dfh):
        dfh.to_parquet(out_dir / "metrics_recordings.parquet", index=False)
        numeric = [c for c in dfh.columns if dfh[c].dtype.kind in "fi"]
        dfh.groupby("subject", as_index=False)[numeric].mean().to_parquet(
            out_dir / "metrics_subjects.parquet", index=False)
        for k in ("F1", "precision", "recall", "accuracy", "missed_rate",
                  "timing_err_ms_median"):
            if k in dfh.columns:
                agg["peak_" + k] = float(dfh[k].mean())
        for k in ("mean_hr_bpm", "rmssd_ms"):
            if f"gt_{k}" in dfh and f"pr_{k}" in dfh:
                agg["MAE_" + k] = float((dfh[f"gt_{k}"] - dfh[f"pr_{k}"]).abs().mean())
    return agg, dfw

  [push_ok] n=1 commit=2023e954b5da24265cd9e6c1a548de9522cc46b7 msg=nb03_baselines_v3 major-stage @ 17:42Z


---
# 6 · Train

The loop below is the whole notebook. For each queued run it builds the split, trains with early
stopping, evaluates on the held-out subjects, writes everything under `runs/<run_id>/`, marks the
run complete and pushes.

Watch the `val` column. If it stops improving in the first few epochs across every model, the
targets are probably misaligned — stop and check NB02's `nb02_fig1_window.png`.

**You can interrupt at any point.** The current epoch's checkpoint is already on disk, the
interrupt handler pushes it, and the next session resumes mid-run.

In [9]:
# Write the isolated worker outside MODULES: it is orchestration infrastructure and therefore
# does not alter the scientific library/config hash of checkpoints that already exist on HF.
WORKER_SRC = r"""
import argparse
import gc
import hashlib
import json
import os
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def current_rss_gb():
    try:
        for line in Path("/proc/self/status").read_text().splitlines():
            if line.startswith("VmRSS:"):
                return float(line.split()[1]) / 2**20
    except Exception:
        pass
    return float("nan")


def release_memory():
    gc.collect()
    try:
        import pyarrow as pa
        pa.default_memory_pool().release_unused()
    except Exception:
        pass
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def close_dataset(ds):
    base = getattr(ds, "base", ds)
    cache = getattr(base, "_cache", None)
    if not isinstance(cache, dict):
        return
    for rec in list(cache.values()):
        data = getattr(rec, "data", None)
        try:
            if hasattr(data, "close"):
                data.close()
            mmap = getattr(data, "_mmap", None)
            if mmap is not None and not getattr(mmap, "closed", False):
                mmap.close()
        except Exception:
            pass
    cache.clear()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--context", required=True)
    ap.add_argument("--run-id", required=True)
    ap.add_argument("--experiment", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--fold", required=True, type=int)
    args = ap.parse_args()

    ctx = json.loads(Path(args.context).read_text(encoding="utf-8"))
    cfg = ctx["cfg"]
    work = Path(ctx["work"])
    data = Path(ctx["data"])
    out = work / "runs" / args.run_id
    out.mkdir(parents=True, exist_ok=True)
    status_path = work / "worker_status.json"
    sys.path.insert(0, str(work))

    from crvs_data import WindowDataset, FS
    from crvs_engine import Trainer, seed_all
    from crvs_losses import MSEOnly, CompositeLoss
    from crvs_metrics import seg_metrics, detect_r_peaks, hrv_from_peaks, peak_detection_scores
    from crvs_models import build_baseline, count_params

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

    # A child never needs pinned pages or loader subprocesses: its lifetime is deliberately
    # short, and mmap reads are already much faster than GPU compute.
    base_loader = Trainer._loader
    base_validation = Trainer._validation
    base_write_epoch = Trainer._write_epoch

    def isolated_loader(self, *a, **kw):
        loader = base_loader(self, *a, **kw)
        loader.pin_memory = False
        return loader

    def isolated_validation(self, *a, **kw):
        try:
            return base_validation(self, *a, **kw)
        finally:
            release_memory()

    def migrate_metric_names(record):
        for old, new in (("train_mse", "train_loss_mse"),
                         ("val_mse", "val_loss_mse")):
            if old in record:
                record.setdefault(new, record[old])
                record.pop(old, None)

    def portable_write_epoch(self, rec):
        for old_rec in self.state.get("history", []):
            migrate_metric_names(old_rec)
        migrate_metric_names(rec)
        base_write_epoch(self, rec)

    Trainer._loader = isolated_loader
    Trainer._validation = isolated_validation
    Trainer._write_epoch = portable_write_epoch

    class TargetConventionDataset:
        def __init__(self, base, target_01=False):
            self.base = base
            self.target_01 = bool(target_01)
            self.index = base.index
        def __len__(self):
            return len(self.base)
        def set_epoch(self, epoch):
            self.base.set_epoch(epoch)
        def __getitem__(self, i):
            x, y, pk, rr = self.base[i]
            if self.target_01:
                y = (y + 1.0) * 0.5
            return x, y, pk, rr

    class BaselineOutputConvention(torch.nn.Module):
        def __init__(self, base, target_01=False):
            super().__init__()
            self.base = base
            self.target_01 = bool(target_01)
        def forward(self, x):
            pred = self.base(x)
            if self.target_01:
                pred["wave"] = (pred["wave"] + 1.0) * 0.5
                if "aux" in pred:
                    pred["aux"] = [torch.sigmoid(v) for v in pred["aux"]]
            return pred

    W = pd.read_parquet(data / "windows.parquet")
    norm_all = json.loads((data / "norm_stats.json").read_text(encoding="utf-8"))
    expinfo = json.loads((data / "experiments.json").read_text(encoding="utf-8"))
    experiments = expinfo["experiments"]
    rec_dir = data / "recordings"

    sub = W[W["scenario_canon"].isin(experiments[args.experiment])]
    te_g, va_g = args.fold % cfg["N_FOLDS"], (args.fold + 1) % cfg["N_FOLDS"]
    tri = sub[~sub["fold_group"].isin([te_g, va_g])]
    vai = sub[(sub["fold_group"] == va_g) & sub["no_overlap"]]
    tei = sub[(sub["fold_group"] == te_g) & sub["no_overlap"]]
    if set(tri["subject"]) & set(tei["subject"]):
        raise RuntimeError("SUBJECT LEAK")
    norm = norm_all.get(f"{args.experiment}|{args.fold}")
    if norm is None:
        raise RuntimeError(f"no normalisation stats for {args.experiment}|{args.fold}")
    channel_idx = [expinfo["channels"].index(c) for c in cfg["CHANNELS"]]
    sub_norm = {"mean": [norm["mean"][i] for i in channel_idx],
                "std": [norm["std"][i] for i in channel_idx]}

    def make_dataset(frame, augment):
        base = WindowDataset(rec_dir, frame, sub_norm, cfg["CHANNELS"], augment=augment,
                             seed=cfg["SEED"] + args.fold)
        return TargetConventionDataset(base, cfg["TARGET_01"])

    tr_ds, va_ds, te_ds = (make_dataset(tri, True), make_dataset(vai, False),
                            make_dataset(tei, False))

    def evaluate(Y, P):
        rows = []
        idx_frame = tei.reset_index(drop=True)
        subjects = idx_frame["subject"].to_numpy()
        scenarios = idx_frame["scenario_canon"].to_numpy()
        for i in range(len(Y)):
            metric = seg_metrics(Y[i], P[i], FS)
            metric["subject"] = subjects[i] if i < len(subjects) else "?"
            metric["scenario"] = scenarios[i] if i < len(scenarios) else "?"
            rows.append(metric)
        dfw = pd.DataFrame(rows)
        dfw.to_parquet(out / "metrics_windows.parquet", index=False)
        numeric = [c for c in dfw.columns if dfw[c].dtype.kind in "fi"]
        agg = {c: float(dfw[c].mean()) for c in numeric}
        agg.update({c + "_std": float(dfw[c].std()) for c in numeric})
        hr_rows = []
        for rec_id, group in idx_frame.assign(_row=np.arange(len(idx_frame))).groupby("rec_id"):
            group = group.sort_values("start")
            pos = group["_row"].to_numpy()
            if len(pos) < 2:
                continue
            yg, pg = np.concatenate(Y[pos]), np.concatenate(P[pos])
            gt_hrv = hrv_from_peaks(detect_r_peaks(yg, FS), FS)
            pr_hrv = hrv_from_peaks(detect_r_peaks(pg, FS), FS)
            peak = peak_detection_scores(yg, pg, FS)
            hr_rows.append({"rec_id": rec_id, "subject": str(group["subject"].iloc[0]),
                            "scenario": str(group["scenario_canon"].iloc[0]),
                            **{f"gt_{k}": v for k, v in gt_hrv.items()},
                            **{f"pr_{k}": v for k, v in pr_hrv.items()}, **peak})
        dfh = pd.DataFrame(hr_rows)
        if len(dfh):
            dfh.to_parquet(out / "metrics_recordings.parquet", index=False)
            numeric = [c for c in dfh.columns if dfh[c].dtype.kind in "fi"]
            dfh.groupby("subject", as_index=False)[numeric].mean().to_parquet(
                out / "metrics_subjects.parquet", index=False)
            for key in ("F1", "precision", "recall", "accuracy", "missed_rate",
                        "timing_err_ms_median"):
                if key in dfh:
                    agg["peak_" + key] = float(dfh[key].mean())
            for key in ("mean_hr_bpm", "rmssd_ms"):
                if f"gt_{key}" in dfh and f"pr_{key}" in dfh:
                    agg["MAE_" + key] = float(
                        (dfh[f"gt_{key}"] - dfh[f"pr_{key}"]).abs().mean())
        return agg

    tr = None
    try:
        print(f"[isolated worker] {args.run_id} | pid={os.getpid()} | "
              f"start RSS={current_rss_gb():.2f} GB", flush=True)
        print(f"  train {len(tr_ds):,} | val {len(va_ds):,} | test {len(te_ds):,}",
              flush=True)
        seed_all(cfg["SEED"] + args.fold)
        base = build_baseline(args.model, in_ch=len(cfg["CHANNELS"]), out_ch=1,
                              base=cfg["BASE"], levels=cfg["LEVELS"])
        model = BaselineOutputConvention(base, cfg["TARGET_01"])
        loss_fn = MSEOnly() if cfg["LOSS"] == "mse" else CompositeLoss()
        run_config = {"protocol_id": cfg["PROTOCOL_ID"], "experiment": args.experiment,
                      "model": args.model, "fold": args.fold, "epochs": cfg["EPOCHS"],
                      "channels": cfg["CHANNELS"], "target_01": cfg["TARGET_01"],
                      "base": cfg["BASE"], "levels": cfg["LEVELS"], "loss": cfg["LOSS"],
                      "lr": cfg["LR"], "batch": cfg["BATCH"],
                      "seed": cfg["SEED"] + args.fold,
                      "data_index_sha256": ctx["data_hash"],
                      "library_sha256": ctx["module_hashes"],
                      "train_subjects": sorted(map(str, tri["subject"].unique())),
                      "val_subjects": sorted(map(str, vai["subject"].unique())),
                      "test_subjects": sorted(map(str, tei["subject"].unique()))}
        tr = Trainer(model, loss_fn, out, args.run_id, sync=None, lr=cfg["LR"],
                     weight_decay=cfg["WEIGHT_DECAY"], epochs=cfg["EPOCHS"],
                     patience=cfg["PATIENCE"], batch_size=cfg["BATCH"],
                     num_workers=0, amp=cfg["AMP"], multi_gpu=cfg["MULTI_GPU"],
                     log_every=cfg["LOG_EVERY"],
                     checkpoint_every_steps=cfg["CHECKPOINT_EVERY_STEPS"],
                     checkpoint_every_s=cfg["CHECKPOINT_EVERY_S"],
                     seed=cfg["SEED"] + args.fold,
                     require_dual_gpu=cfg["REQUIRE_DUAL_T4"], run_config=run_config)
        tr.load()
        for rec in tr.state.get("history", []):
            migrate_metric_names(rec)
        if tr.state.get("stop_reason") in ("host_memory_guard", "process_chunk_boundary"):
            tr.state.pop("stop_reason", None)
            if str(tr.state.get("last_error", "")).startswith("HostMemoryGuard:"):
                tr.state.pop("last_error", None)

        full_epochs = int(cfg["EPOCHS"])
        start_epoch = int(tr.state.get("epoch", 0))
        already_finished = (start_epoch >= full_epochs or
                            tr.state.get("stop_reason") == "early_stopping")
        if not already_finished:
            chunk_end = min(full_epochs, start_epoch + int(cfg["EPOCHS_PER_PROCESS"]))
            print(f"  isolated epoch chunk: {start_epoch + 1}..{chunk_end} / {full_epochs}",
                  flush=True)
            tr.epochs = chunk_end
            tr.fit(tr_ds, va_ds)

        naturally_finished = (int(tr.state.get("epoch", 0)) >= full_epochs or
                              tr.state.get("stop_reason") == "early_stopping")
        if not naturally_finished:
            tr.epochs = full_epochs
            tr.state["done"] = False
            tr.state.pop("finished_utc", None)
            tr.state["stop_reason"] = "process_chunk_boundary"
            tr.save("state", reason="process-chunk-boundary")
            atomic_json(status_path, {"status": "chunk_complete", "run_id": args.run_id,
                                      "epoch": int(tr.state["epoch"]),
                                      "rss_gb": current_rss_gb(),
                                      "pid": os.getpid(),
                                      "finished_utc": datetime.now(timezone.utc).isoformat()})
            print(f"[isolated worker] chunk saved at epoch {tr.state['epoch']}; exiting", flush=True)
            return 0

        tr.epochs = full_epochs
        Y, P = tr.predict(te_ds)
        agg = evaluate(Y, P)
        keep = min(200, len(Y))
        sel = np.linspace(0, len(Y) - 1, keep).astype(int)
        np.savez_compressed(out / "preds_sample.npz", y=Y[sel].astype(np.float32),
                            p=P[sel].astype(np.float32),
                            subject=tei["subject"].to_numpy()[sel].astype(str))
        budget = ctx.get("budget_by_model", {}).get(args.model, {})
        summary = {"run_id": args.run_id, "protocol_id": cfg["PROTOCOL_ID"],
                   "config_hash": tr.config_hash,
                   "memory_runtime_version": ctx["memory_runtime_version"],
                   "process_isolated": True, "epochs_per_process": cfg["EPOCHS_PER_PROCESS"],
                   "data_loader_workers": 0, "pin_memory": False,
                   "experiment": args.experiment, "model": args.model, "fold": args.fold,
                   "channels": cfg["CHANNELS"], "target_01": cfg["TARGET_01"],
                   "loss": cfg["LOSS"], "epochs_run": tr.state["epoch"],
                   "best_epoch": tr.state["best_epoch"], "best_val": tr.state["best"],
                   "params": count_params(tr.raw_model),
                   "gflops_per_window": budget.get("gflops_per_window"),
                   "forward_ms_batch4": budget.get("forward_ms_batch4"),
                   "n_train": len(tr_ds), "n_val": len(va_ds), "n_test": len(te_ds),
                   "test_subjects": sorted(map(str, tei["subject"].unique())),
                   "metrics": agg, "finished_utc": datetime.now(timezone.utc).isoformat()}
        atomic_json(out / "summary.json", summary)
        atomic_json(status_path, {"status": "run_complete", "run_id": args.run_id,
                                  "epoch": int(tr.state["epoch"]), "metrics": agg,
                                  "rss_gb": current_rss_gb(), "pid": os.getpid(),
                                  "finished_utc": datetime.now(timezone.utc).isoformat()})
        print(f"  --> CC_t {agg['CC_temporal']:.2f}  CC_s {agg['CC_spectral']:.2f}  "
              f"MAE {agg['MAE']:.5f}  MSE {agg['MSE']:.5f}", flush=True)
        return 0
    except KeyboardInterrupt:
        if tr is not None and getattr(tr, "_active", False):
            try:
                tr.emergency_checkpoint()
            except Exception:
                pass
        atomic_json(status_path, {"status": "interrupted", "run_id": args.run_id,
                                  "rss_gb": current_rss_gb(), "pid": os.getpid(),
                                  "finished_utc": datetime.now(timezone.utc).isoformat()})
        return 130
    except Exception as exc:
        (out / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        atomic_json(status_path, {"status": "failed", "run_id": args.run_id,
                                  "error": f"{type(exc).__name__}: {exc}",
                                  "rss_gb": current_rss_gb(), "pid": os.getpid(),
                                  "finished_utc": datetime.now(timezone.utc).isoformat()})
        traceback.print_exc()
        return 1
    finally:
        for ds in (locals().get("tr_ds"), locals().get("va_ds"), locals().get("te_ds")):
            if ds is not None:
                close_dataset(ds)
        release_memory()


if __name__ == "__main__":
    raise SystemExit(main())
"""
WORKER_PATH = WORK / "nb03_run_worker.py"
WORKER_PATH.write_text(WORKER_SRC, encoding="utf-8")
WORKER_CONTEXT = WORK / "nb03_worker_context.json"
WORKER_CONTEXT.write_text(json.dumps({
    "cfg": CFG, "work": str(WORK), "data": str(DATA), "data_hash": DATA_HASH,
    "module_hashes": MODULE_HASHES, "budget_by_model": BUDGET_BY_MODEL,
    "memory_runtime_version": MEMORY_RUNTIME_VERSION,
}, indent=2, default=str), encoding="utf-8")

t_start = time.time()
budget = CFG["TIME_BUDGET_H"] * 3600
completed_now = []
failed_now = []
session_stop_reason = None
_active_worker = {"proc": None, "run_id": None}

def _stop_active_worker():
    proc = _active_worker.get("proc")
    if proc is None or proc.poll() is not None:
        return
    print(f"\n  stopping isolated worker {_active_worker.get('run_id')} safely...", flush=True)
    try:
        proc.send_signal(signal.SIGINT)
        proc.wait(timeout=120)
    except Exception:
        # Do not force-kill here: the parent HF interrupt hook will upload the last atomic
        # epoch checkpoint even if the child needs the platform to terminate it.
        pass

def _pull_active_checkpoint(rid, out):
    if (out / "state.json").exists() and not (out / "state.pt").exists():
        print("  interrupted remote run found; restoring exact checkpoint...")
        return sync.pull(allow_patterns=[f"runs/{rid}/state.pt", f"runs/{rid}/best.pt",
                                         f"runs/{rid}/state.json", f"runs/{rid}/run_config.json",
                                         f"runs/{rid}/environment.json", f"runs/{rid}/*.jsonl",
                                         f"runs/{rid}/*.csv", f"runs/{rid}/validation_windows/*",
                                         f"runs/{rid}/validation_recordings/*"])
    return True

def _launch_chunk(q):
    rid = q["run_id"]
    status_path = WORK / "worker_status.json"
    if status_path.exists():
        status_path.unlink()
    cmd = [sys.executable, "-u", str(WORKER_PATH), "--context", str(WORKER_CONTEXT),
           "--run-id", rid, "--experiment", q["exp"], "--model", q["model"],
           "--fold", str(q["fold"])]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    sync.mark_dirty(f"{rid}:isolated-worker-active")
    proc = subprocess.Popen(cmd, cwd=str(WORK), env=env)
    _active_worker.update(proc=proc, run_id=rid)
    sync.set_before_final_flush(_stop_active_worker)
    try:
        rc = proc.wait()
    finally:
        _active_worker.update(proc=None, run_id=None)
        sync.set_before_final_flush(None)
    try:
        status = json.loads(status_path.read_text(encoding="utf-8"))
    except Exception:
        status = {"status": "failed", "run_id": rid,
                  "error": f"worker exited {rc} without a status file"}
    status["return_code"] = rc
    sync.mark_dirty(f"{rid}:isolated-worker-{status.get('status')}")
    parent_ram = release_runtime_memory(f"worker {rid} exited", aggressive=True)
    sync.log("worker_process_reaped", run_id=rid, pid=status.get("pid"),
             worker_status=status.get("status"), epoch=status.get("epoch"),
             worker_rss_gb=status.get("rss_gb"),
             parent_rss_gb=parent_ram.get("process_rss_gb"),
             policy=MEMORY_RUNTIME_VERSION)
    return status

if not CFG.get("PROCESS_ISOLATION", True):
    raise RuntimeError("PROCESS_ISOLATION must stay enabled for this Kaggle workload")

for qi, q in enumerate(list(todo), 1):
    rid = q["run_id"]
    out = WORK / "runs" / rid
    out.mkdir(parents=True, exist_ok=True)
    if not _pull_active_checkpoint(rid, out):
        failed_now.append({"run_id": rid, "error": "remote checkpoint download failed"})
        continue
    print("\n" + "=" * 78)
    print(f"[{qi}/{len(todo)}] {rid} — isolated process chunks of "
          f"{CFG['EPOCHS_PER_PROCESS']} epoch(s)")
    print("=" * 78)
    while True:
        if time.time() - t_start >= budget:
            session_stop_reason = "time_budget"
            print(f"\n=== {CFG['TIME_BUDGET_H']:.2f} h budget reached; pushing and pausing. ===")
            break
        try:
            status = _launch_chunk(q)
        except KeyboardInterrupt:
            _stop_active_worker()
            sync.mark_dirty(f"{rid}:parent-interrupt")
            sync.flush(final=True, force=True, msg=f"{rid} interrupted; isolated checkpoint")
            print("\ninterrupted — the latest atomic child checkpoint was pushed.")
            raise
        state = status.get("status")
        if state == "chunk_complete":
            print(f"  worker exited cleanly at epoch {status.get('epoch')}; "
                  "OS RAM reclaimed; launching the next chunk.")
            continue
        if state == "run_complete" and (out / "summary.json").exists():
            summary = json.loads((out / "summary.json").read_text(encoding="utf-8"))
            metrics = summary.get("metrics", {})
            done.add(rid); done_all.add(rid); completed_now.append(rid)
            STATE["completed"] = sorted(done_all); sync.save_state(STATE)
            pushed = sync.flush(force=True, msg=(
                f"{rid} complete CCt={metrics.get('CC_temporal', float('nan')):.2f}"))
            if pushed:
                for name in ("state.pt", "best.pt"):
                    checkpoint = out / name
                    if checkpoint.exists():
                        checkpoint.unlink()
                sync.log("local_checkpoints_pruned", run_id=rid, remote_copy=True)
            break
        error = status.get("error", f"worker returned {status.get('return_code')}")
        print(f"  !! isolated worker failed: {error}")
        failed_now.append({"run_id": rid, "error": error})
        sync.flush(force=True, msg=f"{rid} worker failure checkpoint")
        break
    if session_stop_reason:
        break

print(f"\ncompleted this session: {len(completed_now)}   total: {len(done)}/{len(QUEUE)}")
print("Each worker process has exited; its RAM and CUDA context were reclaimed by Linux.")

# The legacy in-kernel loop remains below only as a source-readable fallback.  Emptying its
# input guarantees it cannot execute in this process-isolated notebook.
todo = []
MAJOR("02_training_isolated")


[1/5] A_apnea__multireslinknet__f0 — isolated process chunks of 5 epoch(s)
  [dirty] reason=A_apnea__multireslinknet__f0:isolated-worker-active
[isolated worker] A_apnea__multireslinknet__f0 | pid=285 | start RSS=0.64 GB
  train 676 | val 117 | test 105
  isolated epoch chunk: 1..5 / 120


/kaggle/working/nb03/crvs_engine.py:360: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  "global_step": self.state["global_step"], "loss": float(loss),


  ep   1/5 train 0.05626 val 0.06271 CCt 2.5 CCs -2.1 * 5s ETA 0.0h
  ep   2/5 train 0.04421 val 0.04845 CCt 6.3 CCs -0.9 * 3s ETA 0.0h
  ep   3/5 train 0.04196 val 0.04522 CCt 14.7 CCs 28.1 * 3s ETA 0.0h
  ep   4/5 train 0.04186 val 0.04925 CCt -7.2 CCs 35.4  3s ETA 0.0h
  ep   5/5 train 0.04072 val 0.04594 CCt 17.8 CCs 33.1  3s ETA 0.0h
[isolated worker] chunk saved at epoch 5; exiting
  [dirty] reason=A_apnea__multireslinknet__f0:isolated-worker-chunk_complete
  RAM after worker A_apnea__multireslinknet__f0 exited: 1.43 GB RSS | 29.06 GB host available
  [worker_process_reaped] run_id=A_apnea__multireslinknet__f0 pid=285 worker_status=chunk_complete epoch=5 worker_rss_gb=2.339733123779297 parent_rss_gb=1.4347038269042969 policy=nb03-process-isolated-v1
  worker exited cleanly at epoch 5; OS RAM reclaimed; launching the next chunk.
  [dirty] reason=A_apnea__multireslinknet__f0:isolated-worker-active
[isolated worker] A_apnea__multireslinknet__f0 | pid=448 | start RSS=0.64 GB
  train 

/kaggle/working/nb03/crvs_engine.py:360: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  "global_step": self.state["global_step"], "loss": float(loss),


  ep   6/10 train 0.04041 val 0.04546 CCt 17.2 CCs 47.9  5s ETA 0.0h
  ep   7/10 train 0.03912 val 0.04697 CCt 15.6 CCs 80.1  3s ETA 0.0h
  ep   8/10 train 0.03797 val 0.04441 CCt 25.6 CCs 90.4 * 3s ETA 0.0h
  ep   9/10 train 0.03634 val 0.04170 CCt 31.3 CCs 95.1 * 3s ETA 0.0h
  ep  10/10 train 0.03447 val 0.04364 CCt 31.1 CCs 96.3  3s ETA 0.0h
[isolated worker] chunk saved at epoch 10; exiting
  [dirty] reason=A_apnea__multireslinknet__f0:isolated-worker-chunk_complete
  RAM after worker A_apnea__multireslinknet__f0 exited: 1.43 GB RSS | 29.12 GB host available
  [worker_process_reaped] run_id=A_apnea__multireslinknet__f0 pid=448 worker_status=chunk_complete epoch=10 worker_rss_gb=2.3278236389160156 parent_rss_gb=1.4347114562988281 policy=nb03-process-isolated-v1
  worker exited cleanly at epoch 10; OS RAM reclaimed; launching the next chunk.
  [dirty] reason=A_apnea__multireslinknet__f0:isolated-worker-active
[isolated worker] A_apnea__multireslinknet__f0 | pid=611 | start RSS=0.64 G

...

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


  RAM after worker A_apnea__multireslinknet__f4 exited: 1.45 GB RSS | 29.02 GB host available
  [worker_process_reaped] run_id=A_apnea__multireslinknet__f4 pid=7013 worker_status=run_complete epoch=33 worker_rss_gb=2.291717529296875 parent_rss_gb=1.4528656005859375 policy=nb03-process-isolated-v1


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [push_ok] n=6 commit=49262c5eaa7f0c214946d3a51b15e9345d425d48 msg=A_apnea__multireslinknet__f4 complete CCt=39.15
  [local_checkpoints_pruned] run_id=A_apnea__multireslinknet__f4 remote_copy=True

completed this session: 5   total: 80/80
Each worker process has exited; its RAM and CUDA context were reclaimed by Linux.
  [stage_done] stage=02_training_isolated


---
# 7 · Results, next to the published table

The comparison that decides whether we may proceed. Our MultiResLinkNet row on `B_rva` is the one
that matters: the baseline reports **CC_temporal 61.86, CC_spectral 79.96, MAE 0.14841,
RRMSE_t 0.44618**.

Read the gap in the right direction. Because our splits are subject-wise with non-overlapping test
windows, landing **somewhat below** their figures is the expected and honest outcome — it means we
removed the leakage, not that our implementation is worse. What must hold is the **ordering**:
MultiResLinkNet should still beat FPN, UNet and LinkNet on correlation. If the ordering inverts,
the reimplementation is wrong.

In [10]:
rows = []
ignored_summaries = []
for p in sorted((WORK / "runs").glob("*/summary.json")):
    try:
        s = json.loads(p.read_text())
        if (s.get("run_id") not in queue_ids or
                s.get("protocol_id") != CFG["PROTOCOL_ID"]):
            ignored_summaries.append(s.get("run_id", p.parent.name))
            continue
        rows.append({"experiment": s["experiment"], "model": s["model"], "fold": s["fold"],
                     "run_id": s["run_id"],
                     "params": s.get("params"), "best_epoch": s.get("best_epoch"),
                     **{k: v for k, v in s["metrics"].items() if not k.endswith("_std")}})
    except Exception:
        pass
R = pd.DataFrame(rows)
flat = pd.DataFrame(); CMP = pd.DataFrame()
GATE = {"protocol_id": CFG["PROTOCOL_ID"], "complete": False, "passed": False,
        "expected_b_rva_runs": len(CFG["MODELS"]) * CFG["N_FOLDS"],
        "completed_b_rva_runs": 0,
        "updated_utc": datetime.now(timezone.utc).isoformat()}
if ignored_summaries:
    print(f"ignored {len(ignored_summaries)} non-canonical summary file(s): "
          f"{ignored_summaries[:5]}{' ...' if len(ignored_summaries) > 5 else ''}\n")
if not len(R):
    print("no canonical completed runs yet -- run the training cell.")
else:
    R.to_csv(WORK / "results" / "runs_raw.csv", index=False)
    cols = ["MAE", "MSE", "CC_temporal", "CC_spectral", "RRMSE_temporal", "RRMSE_spectral"]
    agg = (R.groupby(["experiment", "model"])[cols + ["params"]]
             .agg(["mean", "std"]).round(5))
    print("=" * 100); print("OUR RUNS  (mean +/- std across folds)"); print("=" * 100)
    flat = R.groupby(["experiment", "model"])[cols].mean().round(5).reset_index()
    nfold = R.groupby(["experiment", "model"]).size().rename("folds").reset_index()
    flat = flat.merge(nfold, on=["experiment", "model"])
    print(flat.to_string(index=False))
    flat.to_csv(WORK / "results" / "runs_summary.csv", index=False)

    PAPER = {
      ("B_rva","fpn"):            dict(MAE=.14316, MSE=.03422, CC_temporal=59.63, CC_spectral=69.53, RRMSE_temporal=.44694, RRMSE_spectral=.83026),
      ("B_rva","unet"):           dict(MAE=.14798, MSE=.03741, CC_temporal=57.65, CC_spectral=68.39, RRMSE_temporal=.45315, RRMSE_spectral=.94118),
      ("B_rva","linknet"):        dict(MAE=.14780, MSE=.03723, CC_temporal=58.69, CC_spectral=70.91, RRMSE_temporal=.45487, RRMSE_spectral=.86909),
      ("B_rva","multireslinknet"):dict(MAE=.14841, MSE=.03793, CC_temporal=61.86, CC_spectral=79.96, RRMSE_temporal=.44618, RRMSE_spectral=.73269),
      ("A_resting","fpn"):            dict(CC_temporal=58.37, CC_spectral=71.38),
      ("A_resting","unet"):           dict(CC_temporal=63.10, CC_spectral=74.68),
      ("A_resting","linknet"):        dict(CC_temporal=64.35, CC_spectral=74.37),
      ("A_resting","multireslinknet"):dict(CC_temporal=66.10, CC_spectral=82.44),
      ("A_valsalva","fpn"):            dict(CC_temporal=57.53, CC_spectral=65.97),
      ("A_valsalva","unet"):           dict(CC_temporal=58.38, CC_spectral=68.79),
      ("A_valsalva","linknet"):        dict(CC_temporal=56.63, CC_spectral=66.87),
      ("A_valsalva","multireslinknet"):dict(CC_temporal=60.14, CC_spectral=77.05),
      ("A_apnea","fpn"):            dict(CC_temporal=39.12, CC_spectral=51.26),
      ("A_apnea","unet"):           dict(CC_temporal=56.14, CC_spectral=69.97),
      ("A_apnea","linknet"):        dict(CC_temporal=56.22, CC_spectral=70.35),
      ("A_apnea","multireslinknet"):dict(CC_temporal=55.33, CC_spectral=74.66),
    }
    cmp_rows = []
    for _, r in flat.iterrows():
        p = PAPER.get((r["experiment"], r["model"]), {})
        cmp_rows.append({"experiment": r["experiment"], "model": r["model"], "folds": r["folds"],
                         "CC_t_ours": round(r["CC_temporal"], 2),
                         "CC_t_paper": p.get("CC_temporal"),
                         "CC_t_delta": (round(r["CC_temporal"] - p["CC_temporal"], 2)
                                        if "CC_temporal" in p else None),
                         "CC_s_ours": round(r["CC_spectral"], 2),
                         "CC_s_paper": p.get("CC_spectral"),
                         "MAE_ours": round(r["MAE"], 5), "MAE_paper": p.get("MAE")})
    CMP = pd.DataFrame(cmp_rows)
    print("\n" + "=" * 100); print("OURS vs. PUBLISHED"); print("=" * 100)
    print(CMP.to_string(index=False))
    CMP.to_csv(WORK / "results" / "vs_paper.csv", index=False)

    b_raw = R[R["experiment"] == "B_rva"]
    expected_b = {(m, f) for m in CFG["MODELS"] for f in range(CFG["N_FOLDS"])}
    got_b = set(zip(b_raw["model"], b_raw["fold"]))
    missing_b = sorted(expected_b - got_b)
    GATE["completed_b_rva_runs"] = len(got_b & expected_b)
    GATE["missing"] = [{"model": m, "fold": int(f)} for m, f in missing_b]
    GATE["complete"] = not missing_b
    print("\n" + "-" * 100)
    print("REPRODUCTION GATE  (Experiment B, RVA combined)")
    print("-" * 100)
    if missing_b:
        print(f"  GATE PENDING -- {len(got_b & expected_b)}/{len(expected_b)} canonical runs complete.")
        print("  Missing:", ", ".join(f"{m}/f{f}" for m, f in missing_b))
        print("  A partial or single-model result can never pass this gate.")
    else:
        b = flat[flat["experiment"] == "B_rva"].sort_values("CC_temporal", ascending=False)
        print(f"  ranking by CC_temporal: {' > '.join(b['model'].tolist())}")
        top = b.iloc[0]["model"]
        ok = top == "multireslinknet"
        GATE["passed"] = bool(ok)
        GATE["ranking"] = b["model"].tolist()
        print(f"  MultiResLinkNet ranks first: {ok}")
        mr = b[b["model"] == "multireslinknet"]
        if len(mr):
            v = float(mr.iloc[0]["CC_temporal"])
            GATE["multireslinknet_cc_temporal"] = v
            GATE["published_cc_temporal"] = 61.86
            print(f"  our CC_temporal {v:.2f}  vs published 61.86  (delta {v-61.86:+.2f})")
        print("\n  " + ("GATE PASSED -- ordering reproduced, proceed to NB04."
                        if ok else
                        "GATE NOT PASSED -- MultiResLinkNet is not top after all 20 canonical"
                        "\n  B_rva runs. Check the reimplementation before trusting downstream results."))
(WORK / "results" / "reproduction_gate.json").write_text(
    json.dumps(GATE, indent=2, default=str))
MAJOR("03_results")

OUR RUNS  (mean +/- std across folds)
experiment           model     MAE     MSE  CC_temporal  CC_spectral  RRMSE_temporal  RRMSE_spectral  folds
   A_apnea             fpn 0.15469 0.03711     39.81532     75.63118         0.52456         0.78261      5
   A_apnea         linknet 0.15062 0.03630     36.72704     68.84624         0.51611         0.85030      5
   A_apnea multireslinknet 0.15277 0.03657     41.75307     77.12377         0.52132         0.75752      5
   A_apnea            unet 0.15939 0.03990     20.65021     44.47213         0.55189         0.92124      5
 A_resting             fpn 0.16019 0.04175     47.26226     80.33136         0.52400         0.72313      5
 A_resting         linknet 0.17069 0.04694     45.10603     77.29226         0.57268         0.79565      5
 A_resting multireslinknet 0.16251 0.04119     49.92339     79.86253         0.53006         0.71038      5
 A_resting            unet 0.16924 0.04455     20.47695     29.24395         0.57079         0.922

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
S = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
     "grid": "#D3DADB", "amber": "#8A6212"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": S["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 8.5, "axes.titlesize": 10, "axes.titleweight": "bold"})
FIG = WORK / "figures"

if len(R):
    b = R[R["experiment"] == "B_rva"]
    if len(b):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        order = ["fpn", "unet", "linknet", "multireslinknet"]
        order = [o for o in order if o in set(b["model"])]
        for ax, met, ref in [(axes[0], "CC_temporal",
                              {"fpn":59.63,"unet":57.65,"linknet":58.69,"multireslinknet":61.86}),
                             (axes[1], "CC_spectral",
                              {"fpn":69.53,"unet":68.39,"linknet":70.91,"multireslinknet":79.96})]:
            vals = [b[b["model"] == o][met].mean() for o in order]
            ax.bar(order, vals, color=[S["radar"]] * (len(order) - 1) + [S["ecg"]],
                   edgecolor="white", label="ours")
            ax.plot(order, [ref[o] for o in order], "o--", color=S["ink"], ms=5, lw=1.2,
                    label="published")
            ax.set_title(met + "   (Experiment B, RVA)")
            ax.tick_params(axis="x", rotation=18)
            ax.legend(frameon=False, fontsize=7)
        fig.tight_layout(); fig.savefig(FIG / "nb03_fig1_vs_paper.png"); plt.close(fig)
        print("  wrote nb03_fig1_vs_paper.png")

    fig, ax = plt.subplots(figsize=(9, 3.2))
    for m in sorted(set(R["model"])):
        hs = []
        for p in (WORK / "runs").glob(f"*__{m}__*/state.json"):
            try:
                hs += [(h["epoch"], h["val"]) for h in json.loads(p.read_text())["history"]]
            except Exception:
                pass
        if hs:
            d = pd.DataFrame(hs, columns=["epoch", "val"]).groupby("epoch")["val"].mean()
            ax.plot(d.index, d.values, lw=1.3, label=m)
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("validation loss")
    ax.set_title("Baseline training curves (mean over runs)", loc="left")
    ax.legend(frameon=False, ncol=4, fontsize=7)
    fig.savefig(FIG / "nb03_fig2_curves.png"); plt.close(fig)
    print("  wrote nb03_fig2_curves.png")

    samp = sorted((WORK / "runs").glob("B_rva__*/preds_sample.npz"))
    if samp:
        fig, axes = plt.subplots(len(samp), 1, figsize=(11, 1.9 * len(samp)), sharex=True)
        axes = np.atleast_1d(axes)
        tt = np.arange(1024) / 128.0
        for ax, sp in zip(axes, samp):
            z = np.load(sp)
            k = min(3, len(z["y"]) - 1)
            ax.plot(tt, z["y"][k], lw=1.0, color=S["ink"], label="ground truth")
            ax.plot(tt, z["p"][k], lw=1.0, color=S["ecg"], label="reconstructed", alpha=.85)
            ax.set_ylabel(sp.parent.name.split("__")[1], rotation=0, ha="right",
                          va="center", fontsize=7.5)
            ax.tick_params(labelleft=False)
        axes[0].legend(frameon=False, ncol=2, fontsize=7)
        axes[0].set_title("Reconstructed vs. true ECG — one held-out window per baseline", loc="left")
        axes[-1].set_xlabel("seconds")
        fig.tight_layout(); fig.savefig(FIG / "nb03_fig3_qualitative.png"); plt.close(fig)
        print("  wrote nb03_fig3_qualitative.png")
MAJOR("04_figures")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


  wrote nb03_fig1_vs_paper.png
  wrote nb03_fig2_curves.png


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  wrote nb03_fig3_qualitative.png
  [stage_done] stage=04_figures


In [12]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
card = f"""---
license: cc-by-4.0
tags: [radar, ecg, biosignals, contactless-monitoring, time-series]
pipeline_tag: audio-to-audio
---

# CardioMamba-Net — checkpoints and results

Radar-to-ECG reconstruction on the CR-RVS dataset. Produced by `03_baselines.ipynb` and
`04_cardiomamba_train.ipynb`. Corpus: [`{CFG['SRC_REPO']}`](https://huggingface.co/datasets/{CFG['SRC_REPO']}).

## Layout

```
runs/<experiment>__<model>__f<fold>/
    best.pt                  weights at the best validation epoch
    state.pt                 full resumable state (model, optimiser, scheduler, scaler, RNG)
    summary.json             config + aggregated test metrics
    metrics_windows.parquet  per-window metrics on held-out subjects
    metrics_subjects.parquet per-subject HR / HRV / peak-detection scores
    preds_sample.npz         200 held-out reconstructions for qualitative figures
results/                     cross-run tables, including ours vs. the published numbers
figures/                     comparison and training-curve figures
```

## Protocol

128 Hz, 1024-sample (8 s) windows. Splits **by subject**; test windows do not overlap.
Normalisation statistics are per fold from **training windows only**. Baselines use the paper's
single-channel input (`dy`), [0,1] ECG target, 5 levels, 64 base filters, plain MSE, Adam 5e-4.
Canonical protocol ID: `{CFG['PROTOCOL_ID']}`. Non-canonical/quick summaries are excluded from
all aggregate tables and from the reproduction gate.

Updated {now}.
"""
(WORK / "README.md").write_text(card)
ok = sync.flush(final=True, msg=f"{CFG['RUN_ID']} — {len(done)}/{len(QUEUE)} runs complete")
_reason = globals().get("session_stop_reason")
_failures = globals().get("failed_now", [])
if len(done) == len(QUEUE):
    _session_label = "CANONICAL QUEUE COMPLETE"
elif _reason == "time_budget":
    _session_label = "SESSION PAUSED — TIME BUDGET"
elif _failures:
    _session_label = "SESSION ENDED WITH RUN ERRORS"
else:
    _session_label = "SESSION PAUSED — RUNS REMAIN"
if not ok:
    _session_label += " (FINAL PUSH HAD A PROBLEM)"
print("\n" + "=" * 76)
print("  " + _session_label)
print("=" * 76)
print(f"  repo      : {sync.url}")
print(f"  runs done : {len(done)}/{len(QUEUE)}")
print(f"  this run  : {len(completed_now)} new")
print(f"  failures  : {len(_failures)}")
print(f"  elapsed   : {(time.time()-t_start)/3600:.2f} h")
print("=" * 76)
if len(done) < len(QUEUE):
    print(f"\n  {len(QUEUE)-len(done)} run(s) remain. Start a NEW session and run this notebook")
    print("  again -- it resumes from Hugging Face and skips everything already finished.")
else:
    if GATE.get("passed"):
        print("\n  Canonical queue complete and reproduction gate PASSED.")
        print("  Next: 04_cardiomamba_train.ipynb")
    else:
        print("\n  Canonical queue complete, but the reproduction gate did NOT pass.")
        print("  Do not proceed to NB04 until the result has been reviewed.")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


  [push_ok] n=7 commit=7eb54091758fd88de652450866ed71f5f5d80428 msg=nb03_baselines_v3 major-stage @ 18:06Z


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [push_ok] n=8 commit=84bf3e7992a33aecacfb30ab860872c620ca7699 msg=nb03_baselines_v3 — 80/80 runs complete

  CANONICAL QUEUE COMPLETE
  repo      : https://huggingface.co/Shanmuk4622/cardiomamba-baselines-v2
  runs done : 80/80
  this run  : 5 new
  failures  : 0
  elapsed   : 0.41 h

  Canonical queue complete, but the reproduction gate did NOT pass.
  Do not proceed to NB04 until the result has been reviewed.


---
# 8 · Troubleshooting

**No GPU / very slow epochs** — *Session options → Accelerator → GPU T4 x2*. On CPU an epoch takes
minutes instead of seconds; the notebook runs but the queue will not finish.

**CUDA out of memory** — lower `CFG["BATCH"]` to 32 or 16. MultiResLinkNet with `BASE=64` is the
heaviest of the four. Batch 64 across two T4s is comfortable; batch 128 is not.

**`no normalisation stats for <exp>|<fold>`** — NB02 did not emit that combination, usually because
the split was empty after the quality gate. Re-run NB02, or drop that experiment from
`CFG["EXPERIMENTS"]`.

**`SUBJECT LEAK` assertion** — a genuine bug in fold assignment. Do not bypass it.

**Validation loss flat from epoch 1** — check `nb02_fig1_window.png`. If the ECG and `peak_map`
rows are not aligned, the targets are wrong and no architecture will help.

**Notebook says it allocated too much memory / kernel restarted** — this version defaults to
`CFG["WORKERS"] = 0` and runs at most five epochs in each isolated child process. The child exits
after writing an atomic checkpoint, so Linux—not Python's garbage collector—reclaims all of its
RAM and CUDA state. The parent then launches the next chunk automatically. `worker_process_reaped`
events record worker and parent RSS in HF logs. Do not disable `PROCESS_ISOLATION`, raise
`EPOCHS_PER_PROCESS`, raise `WORKERS`, or enable `PIN_MEMORY` in a long Kaggle session.

**Session ended mid-queue** — expected. Start a new session, run the notebook again, and it
resumes. Progress is per run *and* per epoch.

**Cleanup receipt missing** — run `02b_cleanup_baselines_hf_once.ipynb` once. It removes the old
quick/full baseline artefacts in one audited Hub commit and leaves the receipt required here.

**The gate says pending** — expected until all four `B_rva` models have all five folds. A partial
queue cannot pass the gate and cannot authorize NB04.